In [1]:
!pip install torch-geometric

In [2]:
import torch
import torch_geometric

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
PyG: 2.8.1
GPU available: True
GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "PS_20174392719_1491204439457_log.csv"
)

print("Dataset found:", DATA_PATH.exists())

Dataset found: True


In [5]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (6362620, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

First 5 rows:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
# Check the dataset
print("Dataset shape:", df.shape)

# Display the first 5 rows
display(df.head())

# Check column names
print("\nColumns:")
print(df.columns.tolist())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0



Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [7]:

# Take a small sample for our first graph experiment
sample_df = df.head(1000)

# Count unique sender and receiver accounts
unique_accounts = (
    set(sample_df["nameOrig"]) |
    set(sample_df["nameDest"])
)

print("Transactions in sample:", len(sample_df))
print("Unique accounts:", len(unique_accounts))
print("Fraud transactions:", sample_df["isFraud"].sum())

print("\nTransaction types:")
print(sample_df["type"].value_counts())


Transactions in sample: 1000
Unique accounts: 1597
Fraud transactions: 9

Transaction types:
type
PAYMENT     437
CASH_OUT    230
CASH_IN     183
TRANSFER     99
DEBIT        51
Name: count, dtype: int64


In [8]:

import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "PS_20174392719_1491204439457_log.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


Dataset loaded successfully!
Shape: (6362620, 11)


In [9]:

# Take a small sample for our first graph experiment
sample_df = df.head(1000)

# Count unique sender and receiver accounts
unique_accounts = (
    set(sample_df["nameOrig"]) |
    set(sample_df["nameDest"])
)

print("Transactions in sample:", len(sample_df))
print("Unique accounts:", len(unique_accounts))
print("Fraud transactions:", sample_df["isFraud"].sum())

print("\nTransaction types:")
print(sample_df["type"].value_counts())


Transactions in sample: 1000
Unique accounts: 1597
Fraud transactions: 9

Transaction types:
type
PAYMENT     437
CASH_OUT    230
CASH_IN     183
TRANSFER     99
DEBIT        51
Name: count, dtype: int64


In [10]:

import pandas as pd
import torch

# Use the first 1,000 transactions only
sample_df = df.head(1000)

# Create a unique ID for each account
accounts = pd.Index(
    pd.concat([
        sample_df["nameOrig"],
        sample_df["nameDest"]
    ]).unique()
)

# Map each account to a numeric node ID
account_to_id = {
    account: idx for idx, account in enumerate(accounts)
}

# Convert sender and receiver account names to node IDs
src = sample_df["nameOrig"].map(account_to_id).to_numpy()
dst = sample_df["nameDest"].map(account_to_id).to_numpy()

# Create graph edges: sender -> receiver
edge_index = torch.tensor(
    [src, dst],
    dtype=torch.long
)

# Keep fraud labels for each transaction
edge_labels = torch.tensor(
    sample_df["isFraud"].to_numpy(),
    dtype=torch.long
)

print("Number of nodes:", len(accounts))
print("Edge index shape:", edge_index.shape)
print("Number of transaction labels:", len(edge_labels))
print("Fraud labels:", int(edge_labels.sum()))


Number of nodes: 1597
Edge index shape: torch.Size([2, 1000])
Number of transaction labels: 1000
Fraud labels: 9


/tmp/ipykernel_4297/974050986.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor(


In [11]:

import numpy as np
from torch_geometric.data import Data

# Faster way to create the edge tensor
edge_index = torch.from_numpy(
    np.stack([src, dst], axis=0)
).long()

# Create a graph data object
graph_data = Data(
    edge_index=edge_index,
    num_nodes=len(accounts)
)

print(graph_data)
print("Number of nodes:", graph_data.num_nodes)
print("Number of edges:", graph_data.num_edges)
print("Edge labels shape:", edge_labels.shape)
print("Fraud transactions:", int(edge_labels.sum()))


Data(edge_index=[2, 1000], num_nodes=1597)
Number of nodes: 1597
Number of edges: 1000
Edge labels shape: torch.Size([1000])
Fraud transactions: 9


In [12]:

# Select the first 1,000 transactions
sample_df = df.head(1000).copy()

# Select useful transaction features
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# Convert transaction types into numerical columns
transaction_types = pd.get_dummies(
    sample_df["type"],
    dtype=float
)

# Combine numeric features and transaction-type columns
transaction_features = pd.concat(
    [
        sample_df[numeric_features].reset_index(drop=True),
        transaction_types.reset_index(drop=True)
    ],
    axis=1
)

# Convert features to a PyTorch tensor
edge_attr = torch.tensor(
    transaction_features.to_numpy(dtype="float32"),
    dtype=torch.float32
)

print("Transaction feature shape:", edge_attr.shape)
print("Feature names:", transaction_features.columns.tolist())
print("Contains NaN:", transaction_features.isna().any().any())
print("Contains infinite values:",
      np.isinf(transaction_features.to_numpy()).any())


Transaction feature shape: torch.Size([1000, 11])
Feature names: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
Contains NaN: False
Contains infinite values: False


In [13]:

# Attach transaction features to the graph
graph_data.edge_attr = edge_attr

# Keep transaction labels separately for edge-level prediction
graph_data.edge_label = edge_labels

print(graph_data)
print("Node count:", graph_data.num_nodes)
print("Edge count:", graph_data.num_edges)
print("Edge feature shape:", graph_data.edge_attr.shape)
print("Edge label shape:", graph_data.edge_label.shape)


Data(edge_index=[2, 1000], num_nodes=1597, edge_attr=[1000, 11], edge_label=[1000])
Node count: 1597
Edge count: 1000
Edge feature shape: torch.Size([1000, 11])
Edge label shape: torch.Size([1000])


In [14]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCNFraudDetector(nn.Module):
    def __init__(self, num_node_features, num_edge_features,
                 hidden_dim=32):
        super().__init__()

        # Learn account representations from the graph
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # Combine sender, receiver and transaction information
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + num_edge_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x, edge_index, edge_attr):
        # Create account embeddings
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        h = F.relu(h)

        # Get sender and receiver embeddings for each transaction
        sender_h = h[edge_index[0]]
        receiver_h = h[edge_index[1]]

        # Combine all transaction information
        transaction_h = torch.cat(
            [sender_h, receiver_h, edge_attr], dim=1
        )

        # Predict normal (0) or fraud (1)
        return self.classifier(transaction_h)


In [15]:

# Build simple account features from the sample
account_stats = pd.DataFrame(
    index=accounts,
    columns=[
        "transactions_sent",
        "transactions_received",
        "amount_sent",
        "amount_received"
    ]
).fillna(0.0)

sent = sample_df.groupby("nameOrig").agg(
    transactions_sent=("amount", "size"),
    amount_sent=("amount", "sum")
)

received = sample_df.groupby("nameDest").agg(
    transactions_received=("amount", "size"),
    amount_received=("amount", "sum")
)

account_stats.loc[sent.index, sent.columns] = sent
account_stats.loc[received.index, received.columns] = received

x = torch.tensor(
    account_stats.fillna(0).to_numpy(dtype="float32"),
    dtype=torch.float32
)

print("Node feature shape:", x.shape)
print("Contains NaN:", torch.isnan(x).any().item())


Node feature shape: torch.Size([1597, 4])
Contains NaN: False


/tmp/ipykernel_4297/1876793568.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0.0)


In [16]:

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create the GCN model
model = GCNFraudDetector(
    num_node_features=x.shape[1],
    num_edge_features=edge_attr.shape[1],
    hidden_dim=32
).to(device)

# Move graph data and features to the selected device
x = x.to(device)
edge_index = graph_data.edge_index.to(device)
edge_attr = graph_data.edge_attr.to(device)
edge_labels = graph_data.edge_label.to(device)

print("Device:", device)
print("Model initialized successfully!")


Device: cuda
Model initialized successfully!


In [17]:

# Test the model without training
model.eval()

with torch.no_grad():
    logits = model(x, edge_index, edge_attr)

    predictions = logits.argmax(dim=1)

print("Prediction shape:", predictions.shape)
print("First 10 predictions:", predictions[:10].cpu().tolist())
print("Total predicted transactions:", len(predictions))


Prediction shape: torch.Size([1000])
First 10 predictions: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Total predicted transactions: 1000


In [18]:

from sklearn.model_selection import train_test_split

# Transaction indices, not account/node indices
transaction_indices = np.arange(len(edge_labels))

train_idx, test_idx = train_test_split(
    transaction_indices,
    test_size=0.2,
    random_state=42,
    stratify=edge_labels.cpu().numpy()
)

train_idx = torch.tensor(train_idx, dtype=torch.long, device=device)
test_idx = torch.tensor(test_idx, dtype=torch.long, device=device)

print("Training transactions:", len(train_idx))
print("Testing transactions:", len(test_idx))
print("Training fraud cases:", int(edge_labels[train_idx].sum()))
print("Testing fraud cases:", int(edge_labels[test_idx].sum()))


Training transactions: 800
Testing transactions: 200
Training fraud cases: 7
Testing fraud cases: 2


In [19]:

import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score

# Reinitialize the model so training starts fresh
model = GCNFraudDetector(
    num_node_features=x.shape[1],
    num_edge_features=edge_attr.shape[1],
    hidden_dim=32
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train only on training transaction labels
model.train()

for epoch in range(1, 21):
    optimizer.zero_grad()

    logits = model(x, edge_index, edge_attr)

    loss = F.cross_entropy(
        logits[train_idx],
        edge_labels[train_idx]
    )

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d}, Loss: {loss.item():.4f}")

print("GCN training completed!")


Epoch 05, Loss: 43847.0508
Epoch 10, Loss: 11437.5713
Epoch 15, Loss: 2198.6052
Epoch 20, Loss: 1525.4922
GCN training completed!


In [20]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

model.eval()

with torch.no_grad():
    logits = model(x, edge_index, edge_attr)
    probabilities = torch.softmax(logits, dim=1)[:, 1]
    predictions = logits.argmax(dim=1)

y_true = edge_labels[test_idx].cpu().numpy()
y_pred = predictions[test_idx].cpu().numpy()
y_prob = probabilities[test_idx].cpu().numpy()

print("GCN — SAMPLE TEST RESULTS")
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("PR-AUC:", round(average_precision_score(y_true, y_prob), 4))

if len(set(y_true)) == 2:
    print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
else:
    print("ROC-AUC: Not available — test set has only one class")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


GCN — SAMPLE TEST RESULTS
Precision: 0.0
Recall: 0.0
F1-score: 0.0
PR-AUC: 0.01
ROC-AUC: 0.4975

Confusion Matrix:
[[198   0]
 [  2   0]]


In [21]:

# Check whether feature scales are causing unstable training
print("Node features:")
print("Min:", x.min().item())
print("Max:", x.max().item())

print("\nTransaction features:")
print("Min:", edge_attr.min().item())
print("Max:", edge_attr.max().item())

print("\nTraining labels:")
print("Normal:", int((edge_labels[train_idx] == 0).sum()))
print("Fraud:", int((edge_labels[train_idx] == 1).sum()))

print("\nModel output range:")
model.eval()
with torch.no_grad():
    logits_check = model(x, edge_index, edge_attr)
    print("Min logit:", logits_check.min().item())
    print("Max logit:", logits_check.max().item())


Node features:
Min: 0.0
Max: 19914866.0

Transaction features:
Min: 0.0
Max: 19169204.0

Training labels:
Normal: 793
Fraud: 7

Model output range:
Min logit: -1021037.9375
Max logit: 196143.109375


In [22]:

from sklearn.preprocessing import StandardScaler

# Convert features to NumPy arrays on CPU
x_np = x.detach().cpu().numpy()
edge_attr_np = edge_attr.detach().cpu().numpy()

# Scale node features
node_scaler = StandardScaler()
x_scaled = node_scaler.fit_transform(x_np)

# Scale transaction features
edge_scaler = StandardScaler()
edge_attr_scaled = edge_scaler.fit_transform(edge_attr_np)

# Convert back to PyTorch tensors
x = torch.tensor(
    x_scaled, dtype=torch.float32, device=device
)

edge_attr = torch.tensor(
    edge_attr_scaled, dtype=torch.float32, device=device
)

print("Scaled node feature range:",
      x.min().item(), "to", x.max().item())
print("Scaled transaction feature range:",
      edge_attr.min().item(), "to", edge_attr.max().item())
print("NaN in node features:", torch.isnan(x).any().item())
print("NaN in transaction features:",
      torch.isnan(edge_attr).any().item())


Scaled node feature range: -1.2942341566085815 to 31.753463745117188
Scaled transaction feature range: -0.8810215592384338 to 9.773399353027344
NaN in node features: False
NaN in transaction features: False


In [23]:

import torch
import torch.nn.functional as F

# Start with a fresh model
model = GCNFraudDetector(
    num_node_features=x.shape[1],
    num_edge_features=edge_attr.shape[1],
    hidden_dim=32
).to(device)

# Give more weight to the minority fraud class
train_labels = edge_labels[train_idx]
class_counts = torch.bincount(train_labels, minlength=2).float()

class_weights = class_counts.sum() / (2 * class_counts)
class_weights = class_weights.to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# Use a smaller learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(1, 31):
    model.train()
    optimizer.zero_grad()

    logits = model(x, edge_index, edge_attr)

    loss = criterion(
        logits[train_idx],
        edge_labels[train_idx]
    )

    loss.backward()

    # Reduce the risk of very large gradient updates
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d}, Loss: {loss.item():.4f}")

print("Scaled GCN training completed!")


Epoch 05, Loss: 0.6795
Epoch 10, Loss: 0.6941
Epoch 15, Loss: 0.6631
Epoch 20, Loss: 0.6711
Epoch 25, Loss: 0.6628
Epoch 30, Loss: 0.6741
Scaled GCN training completed!


In [24]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

model.eval()

with torch.no_grad():
    logits = model(x, edge_index, edge_attr)
    probabilities = torch.softmax(logits, dim=1)[:, 1]
    predictions = logits.argmax(dim=1)

y_true = edge_labels[test_idx].cpu().numpy()
y_pred = predictions[test_idx].cpu().numpy()
y_prob = probabilities[test_idx].cpu().numpy()

print("GCN — SCALED SAMPLE TEST RESULTS")
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("PR-AUC:", round(average_precision_score(y_true, y_prob), 4))

if len(set(y_true)) == 2:
    print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
else:
    print("ROC-AUC: Not available — only one class in test set")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


GCN — SCALED SAMPLE TEST RESULTS
Precision: 0.0
Recall: 0.0
F1-score: 0.0
PR-AUC: 0.0268
ROC-AUC: 0.7323

Confusion Matrix:
[[153  45]
 [  2   0]]


In [25]:

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    threshold_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_true, threshold_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, threshold_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_true, threshold_pred, zero_division=0
        ),
        "Predicted frauds": int(threshold_pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Precision,Recall,F1-score,Predicted frauds
0,0.1,0.01,1.0,0.0198,200
1,0.3,0.01,1.0,0.0198,200
2,0.5,0.00,0.0,0.0000,45
3,0.7,0.00,0.0,0.0000,0
4,0.9,0.00,0.0,0.0000,0


In [26]:

from pathlib import Path
import pandas as pd

results_dir = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)
results_dir.mkdir(parents=True, exist_ok=True)

# Save the threshold experiment
threshold_df.to_csv(
    results_dir / "gcn_sample_threshold_results.csv",
    index=False
)

# Save the current prototype's main metrics
sample_results = pd.DataFrame([{
    "Model": "GCN",
    "Split": "Random sample prototype",
    "Sample size": len(sample_df),
    "Training transactions": len(train_idx),
    "Testing transactions": len(test_idx),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1-score": f1_score(y_true, y_pred, zero_division=0),
    "PR-AUC": average_precision_score(y_true, y_prob),
    "ROC-AUC": roc_auc_score(y_true, y_prob)
}])

sample_results.to_csv(
    results_dir / "gcn_sample_prototype_results.csv",
    index=False
)

print("Saved files:")
print(list(results_dir.glob("gcn_sample_*.csv")))


Saved files:
[PosixPath('/content/drive/MyDrive/UPI_Fraud_Project/results/gcn_sample_threshold_results.csv'), PosixPath('/content/drive/MyDrive/UPI_Fraud_Project/results/gcn_sample_prototype_results.csv')]


In [27]:

import pandas as pd
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)

for filename in [
    "gcn_sample_threshold_results.csv",
    "gcn_sample_prototype_results.csv"
]:
    file_path = results_dir / filename
    print(f"\n{filename}")
    print("Exists:", file_path.exists())
    if file_path.exists():
        display(pd.read_csv(file_path))



gcn_sample_threshold_results.csv
Exists: True


,Threshold,Precision,Recall,F1-score,Predicted frauds
0,0.1,0.01,1.0,0.019802,200
1,0.3,0.01,1.0,0.019802,200
2,0.5,0.00,0.0,0.000000,45
3,0.7,0.00,0.0,0.000000,0
4,0.9,0.00,0.0,0.000000,0



gcn_sample_prototype_results.csv
Exists: True


,Model,Split,Sample size,Training transactions,Testing transactions,Precision,Recall,F1-score,PR-AUC,ROC-AUC
0,GCN,Random sample prototype,1000,800,200,0.0,0.0,0.0,0.02681,0.732323


In [28]:

# Define the temporal split using the full dataset
train_mask = df["step"] < 355
test_mask = df["step"] >= 355

print("Training transactions:", int(train_mask.sum()))
print("Testing transactions:", int(test_mask.sum()))

print("\nTraining step range:",
      int(df.loc[train_mask, "step"].min()),
      "to",
      int(df.loc[train_mask, "step"].max()))

print("Testing step range:",
      int(df.loc[test_mask, "step"].min()),
      "to",
      int(df.loc[test_mask, "step"].max()))

print("\nTraining fraud cases:",
      int(df.loc[train_mask, "isFraud"].sum()))
print("Testing fraud cases:",
      int(df.loc[test_mask, "isFraud"].sum()))


Training transactions: 5069097
Testing transactions: 1293523

Training step range: 1 to 354
Testing step range: 355 to 743

Training fraud cases: 3955
Testing fraud cases: 4258


In [29]:

# Keep only transactions from the training period
train_df = df.loc[train_mask].copy()

# Identify accounts observed in training transactions
train_accounts = pd.Index(
    pd.concat([
        train_df["nameOrig"],
        train_df["nameDest"]
    ], ignore_index=True).unique()
)

# Map training accounts to numeric node IDs
train_account_to_id = {
    account: idx for idx, account in enumerate(train_accounts)
}

print("Training transactions:", len(train_df))
print("Unique training accounts:", len(train_accounts))
print("Training fraud cases:", int(train_df["isFraud"].sum()))
print("Training step range:",
      int(train_df["step"].min()), "to", int(train_df["step"].max()))


Training transactions: 5069097
Unique training accounts: 7222188
Training fraud cases: 3955
Training step range: 1 to 354


In [30]:

import numpy as np
import torch

# Convert training sender and receiver accounts to numeric IDs
train_src = train_df["nameOrig"].map(train_account_to_id).to_numpy(
    dtype=np.int32
)
train_dst = train_df["nameDest"].map(train_account_to_id).to_numpy(
    dtype=np.int32
)

# Stack edges in source -> destination format
train_edges_np = np.stack([train_src, train_dst], axis=0)

# Convert to PyTorch Geometric's required index type
train_edge_index = torch.from_numpy(
    train_edges_np.astype(np.int64)
)

print("Training edge index shape:", train_edge_index.shape)
print("Training graph nodes:", len(train_accounts))
print("Training graph edges:", train_edge_index.shape[1])
print("Index data type:", train_edge_index.dtype)


Training edge index shape: torch.Size([2, 5069097])
Training graph nodes: 7222188
Training graph edges: 5069097
Index data type: torch.int64


In [31]:

# Prepare training labels as a compact NumPy array
train_labels_np = train_df["isFraud"].to_numpy(dtype=np.int64)

print("Training labels shape:", train_labels_np.shape)
print("Normal transactions:", int((train_labels_np == 0).sum()))
print("Fraud transactions:", int((train_labels_np == 1).sum()))
print("Label data type:", train_labels_np.dtype)


Training labels shape: (5069097,)
Normal transactions: 5065142
Fraud transactions: 3955
Label data type: int64


In [32]:

# Numeric transaction features
numeric_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# Create numeric features as float32
train_numeric = train_df[numeric_cols].to_numpy(
    dtype=np.float32
)

# Encode transaction type as an integer ID
type_mapping = {
    "CASH_IN": 0,
    "CASH_OUT": 1,
    "DEBIT": 2,
    "PAYMENT": 3,
    "TRANSFER": 4
}

train_type = train_df["type"].map(type_mapping).to_numpy(
    dtype=np.int64
)

print("Numeric feature shape:", train_numeric.shape)
print("Transaction type shape:", train_type.shape)
print("Numeric feature dtype:", train_numeric.dtype)
print("Transaction types:", np.unique(train_type))


Numeric feature shape: (5069097, 6)
Transaction type shape: (5069097,)
Numeric feature dtype: float32
Transaction types: [0 1 2 3 4]


In [33]:

from sklearn.preprocessing import StandardScaler

# Fit the scaler on training data only
train_scaler = StandardScaler()

train_numeric_scaled = train_scaler.fit_transform(
    train_numeric
).astype(np.float32)

print("Scaled feature shape:", train_numeric_scaled.shape)
print("Feature dtype:", train_numeric_scaled.dtype)
print("Contains NaN:", np.isnan(train_numeric_scaled).any())
print("Contains infinity:", np.isinf(train_numeric_scaled).any())
print("Minimum:", train_numeric_scaled.min())
print("Maximum:", train_numeric_scaled.max())


Scaled feature shape: (5069097, 6)
Feature dtype: float32
Contains NaN: False
Contains infinity: False
Minimum: -1.8381333
Maximum: 143.91833


In [34]:

# One-hot encode transaction types
train_type_onehot = np.eye(5, dtype=np.float32)[train_type]

# Combine scaled numeric features with transaction-type features
train_edge_features = np.concatenate(
    [train_numeric_scaled, train_type_onehot],
    axis=1
)

print("Transaction feature shape:", train_edge_features.shape)
print("Feature dtype:", train_edge_features.dtype)
print("Contains NaN:", np.isnan(train_edge_features).any())
print("Contains infinity:", np.isinf(train_edge_features).any())


Transaction feature shape: (5069097, 11)
Feature dtype: float32
Contains NaN: False
Contains infinity: False


In [35]:
# Basic features for each account in the training graph

num_accounts = len(train_accounts)

print("Total training accounts:", num_accounts)
print("Total training transactions:", len(train_df))
print("Training edge index shape:", tuple(train_edge_index.shape))
print("Training labels shape:", train_labels_np.shape)
print("Training transaction features shape:", train_edge_features.shape)

Total training accounts: 7222188
Total training transactions: 5069097
Training edge index shape: (2, 5069097)
Training labels shape: (5069097,)
Training transaction features shape: (5069097, 11)


In [36]:
# Count transactions sent and received by each account

import numpy as np

train_src = train_df["nameOrig"].map(train_account_to_id).to_numpy(dtype=np.int32)
train_dst = train_df["nameDest"].map(train_account_to_id).to_numpy(dtype=np.int32)

sent_count = np.bincount(train_src, minlength=num_accounts)
received_count = np.bincount(train_dst, minlength=num_accounts)

print("Sent count shape:", sent_count.shape)
print("Received count shape:", received_count.shape)
print("Total sent transactions:", sent_count.sum())
print("Total received transactions:", received_count.sum())

Sent count shape: (7222188,)
Received count shape: (7222188,)
Total sent transactions: 5069097
Total received transactions: 5069097


In [37]:
# Total amount sent and received by each account

sent_amount = np.bincount(
    train_src,
    weights=train_df["amount"].to_numpy(dtype=np.float64),
    minlength=num_accounts
)

received_amount = np.bincount(
    train_dst,
    weights=train_df["amount"].to_numpy(dtype=np.float64),
    minlength=num_accounts
)

print("Sent amount shape:", sent_amount.shape)
print("Received amount shape:", received_amount.shape)
print("Sent amounts contain NaN:", np.isnan(sent_amount).any())
print("Received amounts contain NaN:", np.isnan(received_amount).any())
print("Sent amounts contain infinity:", np.isinf(sent_amount).any())
print("Received amounts contain infinity:", np.isinf(received_amount).any())

Sent amount shape: (7222188,)
Received amount shape: (7222188,)
Sent amounts contain NaN: False
Received amounts contain NaN: False
Sent amounts contain infinity: False
Received amounts contain infinity: False


In [38]:
# Combine the four account-level features

node_features_np = np.column_stack([
    sent_count,
    received_count,
    sent_amount,
    received_amount
]).astype(np.float32)

print("Node feature shape:", node_features_np.shape)
print("Node feature dtype:", node_features_np.dtype)
print("Contains NaN:", np.isnan(node_features_np).any())
print("Contains infinity:", np.isinf(node_features_np).any())

Node feature shape: (7222188, 4)
Node feature dtype: float32
Contains NaN: False
Contains infinity: False


In [39]:
from sklearn.preprocessing import StandardScaler

node_scaler = StandardScaler()

node_features_scaled = node_scaler.fit_transform(
    node_features_np
).astype(np.float32)

print("Scaled node feature shape:", node_features_scaled.shape)
print("Feature dtype:", node_features_scaled.dtype)
print("Contains NaN:", np.isnan(node_features_scaled).any())
print("Contains infinity:", np.isinf(node_features_scaled).any())
print("Minimum:", node_features_scaled.min())
print("Maximum:", node_features_scaled.max())

Scaled node feature shape: (7222188, 4)
Feature dtype: float32
Contains NaN: False
Contains infinity: False
Minimum: -1.5283232
Maximum: 298.96646


In [40]:
from torch_geometric.data import Data

graph_data = Data(
    x=torch.from_numpy(node_features_scaled),
    edge_index=train_edge_index,
    edge_attr=torch.from_numpy(train_edge_features),
    edge_label=torch.from_numpy(train_labels_np)
)

print(graph_data)
print("Node features:", tuple(graph_data.x.shape))
print("Edges:", tuple(graph_data.edge_index.shape))
print("Edge features:", tuple(graph_data.edge_attr.shape))
print("Labels:", tuple(graph_data.edge_label.shape))
print("Graph device:", graph_data.x.device)

Data(x=[7222188, 4], edge_index=[2, 5069097], edge_attr=[5069097, 11], edge_label=[5069097])
Node features: (7222188, 4)
Edges: (2, 5069097)
Edge features: (5069097, 11)
Labels: (5069097,)
Graph device: cpu


In [41]:
# Check memory used by graph tensors

tensors = {
    "Node features": graph_data.x,
    "Edge index": graph_data.edge_index,
    "Edge features": graph_data.edge_attr,
    "Edge labels": graph_data.edge_label
}

for name, tensor in tensors.items():
    memory_mb = tensor.numel() * tensor.element_size() / (1024 ** 2)
    print(f"{name}: {memory_mb:.2f} MB")

total_mb = sum(
    tensor.numel() * tensor.element_size()
    for tensor in tensors.values()
) / (1024 ** 2)

print(f"\nTotal graph tensor memory: {total_mb:.2f} MB")

Node features: 110.20 MB
Edge index: 77.35 MB
Edge features: 212.71 MB
Edge labels: 38.67 MB

Total graph tensor memory: 438.93 MB


In [42]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    free_memory, total_memory = torch.cuda.mem_get_info()

    print(f"Total GPU memory: {total_memory / (1024**3):.2f} GB")
    print(f"Free GPU memory: {free_memory / (1024**3):.2f} GB")
    print(
        f"Allocated GPU memory: "
        f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB"
    )
else:
    print("GPU not available. Please enable GPU in Colab settings.")

GPU: Tesla T4
Total GPU memory: 14.56 GB
Free GPU memory: 14.40 GB
Allocated GPU memory: 0.02 GB


In [43]:
import torch_geometric
from torch_geometric.loader import NeighborLoader

print("PyTorch Geometric version:", torch_geometric.__version__)
print("NeighborLoader imported successfully!")

try:
    import pyg_lib
    print("pyg-lib: available")
except ImportError:
    print("pyg-lib: not installed")

try:
    import torch_sparse
    print("torch-sparse: available")
except ImportError:
    print("torch-sparse: not installed")

PyTorch Geometric version: 2.8.1
NeighborLoader imported successfully!
pyg-lib: available
torch-sparse: not installed


In [44]:
%pip install torch-geometric

In [45]:
import torch
import torch_geometric
from torch_geometric.loader import NeighborLoader

print("PyTorch version:", torch.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())
print("NeighborLoader imported successfully!")

try:
    import pyg_lib
    print("pyg-lib: available")
except ImportError:
    print("pyg-lib: not installed")

try:
    import torch_sparse
    print("torch-sparse: available")
except ImportError:
    print("torch-sparse: not installed")

PyTorch version: 2.11.0+cu130
PyTorch Geometric version: 2.8.1
GPU available: True
NeighborLoader imported successfully!
pyg-lib: available
torch-sparse: not installed


In [46]:
print("Graph available:", "graph_data" in globals())
print("Node features available:", "node_features_scaled" in globals())
print("Training labels available:", "train_labels_np" in globals())
print("Training dataframe available:", "train_df" in globals())

if "graph_data" in globals():
    print("\nGraph summary:")
    print(graph_data)

Graph available: True
Node features available: True
Training labels available: True
Training dataframe available: True

Graph summary:
Data(x=[7222188, 4], edge_index=[2, 5069097], edge_attr=[5069097, 11], edge_label=[5069097])


In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/PS_20174392719_1491204439457_log.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dataset loaded successfully!")

Dataset shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
Dataset loaded successfully!


In [49]:
train_mask = df["step"] < 355
test_mask = df["step"] >= 355

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training fraud cases:", int(train_df["isFraud"].sum()))
print("Testing fraud cases:", int(test_df["isFraud"].sum()))

print("Training step range:", train_df["step"].min(), "-", train_df["step"].max())
print("Testing step range:", test_df["step"].min(), "-", test_df["step"].max())

Training rows: 5069097
Testing rows: 1293523
Training fraud cases: 3955
Testing fraud cases: 4258
Training step range: 1 - 354
Testing step range: 355 - 743


In [50]:
import pandas as pd
import numpy as np
import torch

train_accounts = pd.Index(
    pd.concat(
        [train_df["nameOrig"], train_df["nameDest"]],
        ignore_index=True
    ).unique()
)

train_account_to_id = {
    account: idx
    for idx, account in enumerate(train_accounts)
}

num_accounts = len(train_accounts)

print("Training transactions:", len(train_df))
print("Unique training accounts:", num_accounts)
print("Total training fraud cases:", int(train_df["isFraud"].sum()))

Training transactions: 5069097
Unique training accounts: 7222188
Total training fraud cases: 3955


In [51]:
# Create source and destination account IDs
train_src = train_df["nameOrig"].map(train_account_to_id).to_numpy(dtype=np.int32)
train_dst = train_df["nameDest"].map(train_account_to_id).to_numpy(dtype=np.int32)

# Create edge index
train_edge_index = torch.from_numpy(
    np.stack([train_src, train_dst], axis=0).astype(np.int64)
)

# Create transaction labels
train_labels_np = train_df["isFraud"].to_numpy(dtype=np.int64)

print("Edge index shape:", tuple(train_edge_index.shape))
print("Edge index dtype:", train_edge_index.dtype)
print("Labels shape:", train_labels_np.shape)
print("Fraud cases:", int(train_labels_np.sum()))
print("Normal cases:", int((train_labels_np == 0).sum()))

Edge index shape: (2, 5069097)
Edge index dtype: torch.int64
Labels shape: (5069097,)
Fraud cases: 3955
Normal cases: 5065142


In [52]:
# Count sent and received transactions per account
sent_count = np.bincount(train_src, minlength=num_accounts)
received_count = np.bincount(train_dst, minlength=num_accounts)

# Calculate total amounts sent and received
amounts = train_df["amount"].to_numpy(dtype=np.float64)

sent_amount = np.bincount(
    train_src, weights=amounts, minlength=num_accounts
)

received_amount = np.bincount(
    train_dst, weights=amounts, minlength=num_accounts
)

print("Sent count shape:", sent_count.shape)
print("Received count shape:", received_count.shape)
print("Sent amount shape:", sent_amount.shape)
print("Received amount shape:", received_amount.shape)

print("Total sent transactions:", sent_count.sum())
print("Total received transactions:", received_count.sum())

Sent count shape: (7222188,)
Received count shape: (7222188,)
Sent amount shape: (7222188,)
Received amount shape: (7222188,)
Total sent transactions: 5069097
Total received transactions: 5069097


In [53]:
node_features_np = np.column_stack([
    sent_count,
    received_count,
    sent_amount,
    received_amount
]).astype(np.float32)

print("Node feature shape:", node_features_np.shape)
print("Feature dtype:", node_features_np.dtype)
print("Contains NaN:", np.isnan(node_features_np).any())
print("Contains infinity:", np.isinf(node_features_np).any())

Node feature shape: (7222188, 4)
Feature dtype: float32
Contains NaN: False
Contains infinity: False


In [54]:
from sklearn.preprocessing import StandardScaler

node_scaler = StandardScaler()

node_features_scaled = node_scaler.fit_transform(
    node_features_np
).astype(np.float32)

print("Scaled node feature shape:", node_features_scaled.shape)
print("Feature dtype:", node_features_scaled.dtype)
print("Contains NaN:", np.isnan(node_features_scaled).any())
print("Contains infinity:", np.isinf(node_features_scaled).any())

Scaled node feature shape: (7222188, 4)
Feature dtype: float32
Contains NaN: False
Contains infinity: False


In [55]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# Scale numeric features using training data only
train_numeric = train_df[numeric_cols].to_numpy(dtype=np.float32)

train_scaler = StandardScaler()
train_numeric_scaled = train_scaler.fit_transform(
    train_numeric
).astype(np.float32)

# One-hot encode transaction type
type_mapping = {
    "CASH_IN": 0,
    "CASH_OUT": 1,
    "DEBIT": 2,
    "PAYMENT": 3,
    "TRANSFER": 4
}

train_type = train_df["type"].map(type_mapping).to_numpy(dtype=np.int64)
train_type_onehot = np.eye(5, dtype=np.float32)[train_type]

# Combine into 11 transaction features
train_edge_features = np.concatenate(
    [train_numeric_scaled, train_type_onehot],
    axis=1
)

print("Transaction feature shape:", train_edge_features.shape)
print("Feature dtype:", train_edge_features.dtype)
print("Contains NaN:", np.isnan(train_edge_features).any())
print("Contains infinity:", np.isinf(train_edge_features).any())

Transaction feature shape: (5069097, 11)
Feature dtype: float32
Contains NaN: False
Contains infinity: False


In [56]:
import torch
import torch_geometric

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())

try:
    import pyg_lib
    print("pyg-lib: available")
except ImportError:
    print("pyg-lib: not installed")

try:
    import torch_sparse
    print("torch-sparse: available")
except ImportError:
    print("torch-sparse: not installed")

PyTorch: 2.11.0+cu130
PyG: 2.8.1
GPU available: True
pyg-lib: available
torch-sparse: not installed


In [57]:
%pip install pyg-lib -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


In [58]:
import pyg_lib
from torch_geometric.loader import NeighborLoader

print("pyg-lib version:", pyg_lib.__version__)
print("NeighborLoader imported successfully!")
print("GPU available:", torch.cuda.is_available())

pyg-lib version: 0.8.0+pt211cu130
NeighborLoader imported successfully!
GPU available: True


In [59]:

%pip install --force-reinstall --no-deps pyg-lib==0.8.0 -f https://data.pyg.org/whl/torch-2.11.0+cu130.html


Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
  Using cached https://data.pyg.org/whl/torch-2.11.0%2Bcu130/pyg_lib-0.8.0%2Bpt211cu130-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
  Attempting uninstall: pyg-lib
    Found existing installation: pyg_lib 0.8.0+pt211cu130
    Uninstalling pyg_lib-0.8.0+pt211cu130:
      Successfully uninstalled pyg_lib-0.8.0+pt211cu130


In [60]:
%pip install torch-sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


In [61]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader

# Tiny test graph: 6 nodes and 8 directed edges
test_edge_index = torch.tensor([
    [0, 1, 2, 3, 4, 5, 1, 4],
    [1, 2, 3, 4, 5, 0, 4, 2]
], dtype=torch.long)

test_x = torch.randn(6, 4)

test_graph = Data(
    x=test_x,
    edge_index=test_edge_index
)

loader = NeighborLoader(
    test_graph,
    num_neighbors=[2],
    batch_size=2,
    input_nodes=torch.arange(6),
    shuffle=False
)

batch = next(iter(loader))

print("Mini-batch sampler works!")
print("Batch nodes:", batch.num_nodes)
print("Batch edges:", batch.edge_index.shape[1])
print("Seed nodes:", batch.batch_size)

Mini-batch sampler works!
Batch nodes: 3
Batch edges: 2
Seed nodes: 2


In [62]:
%pip install torch-sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


In [63]:
import torch
import pyg_lib
import torch_geometric
import torch_geometric.typing as pyg_typing

print("PyTorch:", torch.__version__)
print("pyg-lib path:", pyg_lib.__file__)
print("PyG version:", torch_geometric.__version__)
print("PyG detects pyg-lib:", pyg_typing.WITH_PYG_LIB)

try:
    from pyg_lib import ops
    print("pyg-lib ops imported successfully")
except Exception as e:
    print("pyg-lib ops error:", repr(e))

PyTorch: 2.11.0+cu130
pyg-lib path: /usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py
PyG version: 2.8.1
PyG detects pyg-lib: True
pyg-lib ops imported successfully


In [64]:
import torch
import pyg_lib
import torch_geometric
import torch_geometric.typing as pyg_typing

print("PyTorch:", torch.__version__)
print("pyg-lib:", getattr(pyg_lib, "__version__", "unknown"))
print("PyG:", torch_geometric.__version__)

print("PyG detects pyg-lib:", pyg_typing.WITH_PYG_LIB)
print("CUDA version:", torch.version.cuda)

print("Available pyg-lib ops:")
print([
    name for name in dir(pyg_lib.ops)
    if "neighbor" in name.lower() or "sample" in name.lower()
][:20])

PyTorch: 2.11.0+cu130
pyg-lib: 0.8.0+pt211cu130
PyG: 2.8.1
PyG detects pyg-lib: True
CUDA version: 13.0
Available pyg-lib ops:
['edge_sample', 'sampled_add', 'sampled_div', 'sampled_mul', 'sampled_sub']


In [65]:
import pyg_lib
import pyg_lib.ops
import torch_geometric.typing as pyg_typing

print("Neighbor sampling operator exists:",
      hasattr(pyg_lib.ops, "neighbor_sample"))

print("PyG detection flag:", pyg_typing.WITH_PYG_LIB)

print("Installed pyg-lib location:", pyg_lib.__file__)

Neighbor sampling operator exists: False
PyG detection flag: True
Installed pyg-lib location: /usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py


In [66]:
%pip show pyg-lib torch-geometric torch

Name: pyg_lib
Version: 0.8.0+pt211cu130
Summary: Low-Level Graph Neural Network Operators for PyG
Home-page: https://pyg.org
Author: 
Author-email: PyG Team <team@pyg.org>
License: Copyright (c) 2022 PyG Team <team@pyg.org>

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT

In [67]:
import pyg_lib
import torch
import torch_geometric.typing as pyg_typing

print("pyg-lib sampler module:", hasattr(pyg_lib, "sampler"))
print("PyG detection flag:", pyg_typing.WITH_PYG_LIB)

if hasattr(pyg_lib, "sampler"):
    print(
        "Sampler functions:",
        [name for name in dir(pyg_lib.sampler)
         if "neighbor" in name.lower() or "sample" in name.lower()]
    )

print(
    "Registered PyG operators:",
    [name for name in dir(torch.ops.pyg)
     if "neighbor" in name.lower() or "sample" in name.lower()]
)

pyg-lib sampler module: True
PyG detection flag: True
Sampler functions: ['hetero_neighbor_sample', 'neighbor_sample']
Registered PyG operators: ['neighbor_sample']


In [68]:
import inspect
import pyg_lib

print("neighbor_sample function:")
print(pyg_lib.sampler.neighbor_sample)

try:
    print("\nFunction signature:")
    print(inspect.signature(pyg_lib.sampler.neighbor_sample))
except (TypeError, ValueError) as e:
    print("Signature unavailable:", e)

print("\nFunction module:")
print(pyg_lib.sampler.neighbor_sample.__module__)

neighbor_sample function:
<function neighbor_sample at 0x7b31b67efc40>

Function signature:
(rowptr: torch.Tensor, col: torch.Tensor, seed: torch.Tensor, num_neighbors: List[int], node_time: Optional[torch.Tensor] = None, edge_time: Optional[torch.Tensor] = None, seed_time: Optional[torch.Tensor] = None, edge_weight: Optional[torch.Tensor] = None, csc: bool = False, replace: bool = False, directed: bool = True, disjoint: bool = False, temporal_strategy: str = 'uniform', return_edge_id: bool = True) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, Optional[torch.Tensor], List[int], List[int]]

Function module:
pyg_lib.sampler


In [69]:
import torch
import pyg_lib

# Tiny graph in CSR format
rowptr = torch.tensor([0, 1, 2, 3, 4], dtype=torch.long)
col = torch.tensor([1, 2, 3, 0], dtype=torch.long)
seed = torch.tensor([0], dtype=torch.long)

try:
    result = pyg_lib.sampler.neighbor_sample(
        rowptr=rowptr,
        col=col,
        seed=seed,
        num_neighbors=[2]
    )

    print("Sampler executed successfully!")
    print("Number of returned outputs:", len(result))

    for i, item in enumerate(result):
        if isinstance(item, torch.Tensor):
            print(f"Output {i}: shape={tuple(item.shape)}, dtype={item.dtype}")
        else:
            print(f"Output {i}: {item}")

except Exception as e:
    print("Sampler test failed:", type(e).__name__, str(e))

Sampler executed successfully!
Number of returned outputs: 6
Output 0: shape=(1,), dtype=torch.int64
Output 1: shape=(1,), dtype=torch.int64
Output 2: shape=(2,), dtype=torch.int64
Output 3: shape=(1,), dtype=torch.int64
Output 4: [1, 1]
Output 5: [1]


In [70]:
import inspect
import torch_geometric.typing as pyg_typing

print("Detection flag:", pyg_typing.WITH_PYG_LIB)

source = inspect.getsource(pyg_typing)

for line_no, line in enumerate(source.splitlines(), start=1):
    if "WITH_PYG_LIB" in line or "import pyg_lib" in line:
        print(f"{line_no}: {line}")

Detection flag: True
46:     import pyg_lib  # noqa
47:     WITH_PYG_LIB = True
95:     WITH_PYG_LIB = False


In [71]:
import inspect
import torch_geometric.typing as pyg_typing

source_lines = inspect.getsource(pyg_typing).splitlines()

print("Relevant PyG detection code:\n")

for start, end in [(35, 60), (80, 100)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(source_lines))):
        print(f"{i + 1}: {source_lines[i]}")

Relevant PyG detection code:


--- Lines 35-60 ---
35:         torch.int64,
36:     }
37: elif not typing.TYPE_CHECKING:  # pragma: no cover
38:     INDEX_DTYPES: Set[torch.dtype] = {
39:         torch.int64,
40:     }
41: 
42: if not hasattr(torch, 'sparse_csc'):
43:     torch.sparse_csc = torch.sparse_coo
44: 
45: try:
46:     import pyg_lib  # noqa
47:     WITH_PYG_LIB = True
48:     WITH_GMM = WITH_PT20 and hasattr(pyg_lib.ops, 'grouped_matmul')
49:     WITH_SEGMM = hasattr(pyg_lib.ops, 'segment_matmul')
50:     if (WITH_SEGMM and 'PYTEST_CURRENT_TEST' in os.environ
51:             and torch.cuda.is_available()):
52:         # NOTE `segment_matmul` is currently bugged on older NVIDIA cards which
53:         # let our GPU tests on CI crash. Try if this error is present on the
54:         # current GPU and disable `WITH_SEGMM`/`WITH_GMM` if necessary.
55:         # TODO Drop this code block once `segment_matmul` is fixed.
56:         try:
57:             x = torch.randn(3, 4, device=

In [72]:
import importlib
import traceback

try:
    import torch_geometric.typing as pyg_typing
    importlib.reload(pyg_typing)

    print("Reloaded detection flag:", pyg_typing.WITH_PYG_LIB)

except Exception:
    traceback.print_exc()

Reloaded detection flag: True


In [73]:
import torch
from torch_geometric.loader import NeighborLoader

print("PyG detection flag:", pyg_typing.WITH_PYG_LIB)

try:
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[2],
        input_nodes=torch.tensor([0, 1], dtype=torch.long),
        batch_size=2,
        shuffle=False,
        num_workers=0
    )

    batch = next(iter(loader))
    print("NeighborLoader test successful!")
    print("Sampled nodes:", batch.num_nodes)
    print("Sampled edges:", batch.edge_index.shape[1])

except Exception as e:
    print("NeighborLoader test failed:")
    print(type(e).__name__, str(e))

PyG detection flag: True
NeighborLoader test successful!
Sampled nodes: 2
Sampled edges: 0


In [74]:
required_vars = [
    "node_features_scaled",
    "train_edge_index",
    "train_edge_features",
    "train_labels_np"
]

for var in required_vars:
    print(f"{var}: {'Available' if var in globals() else 'Missing'}")

node_features_scaled: Available
train_edge_index: Available
train_edge_features: Available
train_labels_np: Available


In [75]:
import torch
from torch_geometric.data import Data

graph_data = Data(
    x=torch.from_numpy(node_features_scaled),
    edge_index=train_edge_index,
    edge_attr=torch.from_numpy(train_edge_features),
    edge_label=torch.from_numpy(train_labels_np)
)

print("Graph created successfully!")
print("Number of nodes:", graph_data.num_nodes)
print("Number of edges:", graph_data.num_edges)
print("Node features shape:", graph_data.x.shape)
print("Edge features shape:", graph_data.edge_attr.shape)
print("Edge labels shape:", graph_data.edge_label.shape)

Graph created successfully!
Number of nodes: 7222188
Number of edges: 5069097
Node features shape: torch.Size([7222188, 4])
Edge features shape: torch.Size([5069097, 11])
Edge labels shape: torch.Size([5069097])


In [76]:
import torch
from torch_geometric.loader import NeighborLoader

loader = NeighborLoader(
    graph_data,
    num_neighbors=[2],
    input_nodes=torch.tensor([0, 1], dtype=torch.long),
    batch_size=2,
    shuffle=False,
    num_workers=0
)

batch = next(iter(loader))

print("NeighborLoader test successful!")
print("Sampled nodes:", batch.num_nodes)
print("Sampled edges:", batch.edge_index.shape[1])
print("Seed nodes:", batch.batch_size)

NeighborLoader test successful!
Sampled nodes: 2
Sampled edges: 0
Seed nodes: 2


In [77]:
import pyg_lib
import torch_geometric.typing as pyg_typing
import torch_geometric.utils._index_sort as index_sort_module

print("Actual pyg_lib module:", pyg_lib)
print("Actual pyg_lib has ops:", hasattr(pyg_lib, "ops"))

print("PyG detection flag:", pyg_typing.WITH_PYG_LIB)
print("Typing module pyg_lib:", pyg_typing.pyg_lib)
print("Index-sort module pyg_lib:", index_sort_module.pyg_lib)

print(
    "Index-sort operator available:",
    hasattr(pyg_lib.ops, "index_sort")
)

Actual pyg_lib module: <module 'pyg_lib' from '/usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py'>
Actual pyg_lib has ops: True
PyG detection flag: True
Typing module pyg_lib: <module 'pyg_lib' from '/usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py'>
Index-sort module pyg_lib: <module 'pyg_lib' from '/usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py'>
Index-sort operator available: True


In [78]:
import pyg_lib
import torch_geometric.utils._index_sort as index_sort_module

index_sort_module.pyg_lib = pyg_lib

print("Index-sort reference refreshed:", index_sort_module.pyg_lib)
print(
    "Index-sort operator available:",
    hasattr(index_sort_module.pyg_lib.ops, "index_sort")
)

Index-sort reference refreshed: <module 'pyg_lib' from '/usr/local/lib/python3.13/dist-packages/pyg_lib/__init__.py'>
Index-sort operator available: True


In [79]:
import torch
from torch_geometric.loader import NeighborLoader

loader = NeighborLoader(
    graph_data,
    num_neighbors=[2],
    input_nodes=torch.tensor([0, 1], dtype=torch.long),
    batch_size=2,
    shuffle=False,
    num_workers=0
)

batch = next(iter(loader))

print("NeighborLoader test successful!")
print("Sampled nodes:", batch.num_nodes)
print("Sampled edges:", batch.edge_index.shape[1])
print("Seed nodes:", batch.batch_size)

NeighborLoader test successful!
Sampled nodes: 2
Sampled edges: 0
Seed nodes: 2


In [80]:
import torch

edge_index = graph_data.edge_index

print("Total nodes:", graph_data.num_nodes)
print("Total directed edges:", edge_index.shape[1])

print("First 10 source nodes:", edge_index[0, :10].tolist())
print("First 10 destination nodes:", edge_index[1, :10].tolist())

test_nodes = torch.tensor([0, 1], dtype=torch.long)

for node in test_nodes.tolist():
    outgoing = (edge_index[0] == node).sum().item()
    incoming = (edge_index[1] == node).sum().item()

    print(
        f"Node {node}: outgoing edges = {outgoing}, "
        f"incoming edges = {incoming}"
    )

Total nodes: 7222188
Total directed edges: 5069097
First 10 source nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First 10 destination nodes: [5063100, 5063101, 5063102, 5063103, 5063104, 5063105, 5063106, 5063107, 5063108, 5063109]
Node 0: outgoing edges = 1, incoming edges = 0
Node 1: outgoing edges = 1, incoming edges = 0


In [81]:
import torch
from torch_geometric.loader import NeighborLoader

# First transaction's destination node
destination_node = graph_data.edge_index[1, 0].item()

print("Destination node:", destination_node)
print(
    "Incoming edges for destination:",
    (graph_data.edge_index[1] == destination_node).sum().item()
)

loader = NeighborLoader(
    graph_data,
    num_neighbors=[2],
    input_nodes=torch.tensor([destination_node], dtype=torch.long),
    batch_size=1,
    shuffle=False,
    num_workers=0
)

batch = next(iter(loader))

print("Sampled nodes:", batch.num_nodes)
print("Sampled edges:", batch.edge_index.shape[1])
print("Seed nodes:", batch.batch_size)

Destination node: 5063100
Incoming edges for destination: 1
Sampled nodes: 2
Sampled edges: 1
Seed nodes: 1


In [82]:
print("Transaction labels:", graph_data.edge_label.shape)
print("Transaction features:", graph_data.edge_attr.shape)
print("Graph edges:", graph_data.edge_index.shape[1])

print("\nFirst 10 transaction labels:")
print(graph_data.edge_label[:10].tolist())

print("\nFirst transaction features:")
print(graph_data.edge_attr[0].tolist())

print("\nFraud transactions in training data:",
      int(graph_data.edge_label.sum().item()))


Transaction labels: torch.Size([5069097])
Transaction features: torch.Size([5069097, 11])
Graph edges: 5069097

First 10 transaction labels:
[0, 0, 1, 1, 0, 0, 0, 0, 0, 0]

First transaction features:
[-1.8381333351135254, -0.2676636874675751, -0.23009628057479858, -0.2379501312971115, -0.34030091762542725, -0.34420230984687805, 0.0, 0.0, 0.0, 1.0, 0.0]

Fraud transactions in training data: 3955


In [83]:
import numpy as np
import torch

print("First 5 source IDs:",
      graph_data.edge_index[0, :5].tolist())

print("First 5 destination IDs:",
      graph_data.edge_index[1, :5].tolist())

print("First 5 transaction labels:",
      graph_data.edge_label[:5].tolist())

print("First 5 transaction feature rows match:",
      torch.allclose(
          graph_data.edge_attr[:5],
          torch.from_numpy(train_edge_features[:5])
      ))

print("First 5 labels match:",
      np.array_equal(
          graph_data.edge_label[:5].numpy(),
          train_labels_np[:5]
      ))

First 5 source IDs: [0, 1, 2, 3, 4]
First 5 destination IDs: [5063100, 5063101, 5063102, 5063103, 5063104]
First 5 transaction labels: [0, 0, 1, 1, 0]
First 5 transaction feature rows match: True
First 5 labels match: True


In [84]:
import torch
import psutil

ram = psutil.virtual_memory()

print(f"System RAM total: {ram.total / (1024**3):.2f} GB")
print(f"System RAM available: {ram.available / (1024**3):.2f} GB")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    free_gpu, total_gpu = torch.cuda.mem_get_info()

    print(f"GPU: {gpu.name}")
    print(f"GPU memory total: {total_gpu / (1024**3):.2f} GB")
    print(f"GPU memory free: {free_gpu / (1024**3):.2f} GB")
else:
    print("GPU not available")

System RAM total: 12.67 GB
System RAM available: 5.94 GB
GPU: Tesla T4
GPU memory total: 14.56 GB
GPU memory free: 14.40 GB


In [85]:
print("Graph is on CPU:", graph_data.x.device.type == "cpu")
print("Edge index dtype:", graph_data.edge_index.dtype)
print("Edge index contiguous:", graph_data.edge_index.is_contiguous())
print("Node features dtype:", graph_data.x.dtype)
print("Edge features dtype:", graph_data.edge_attr.dtype)
print("Training labels dtype:", graph_data.edge_label.dtype)

Graph is on CPU: True
Edge index dtype: torch.int64
Edge index contiguous: True
Node features dtype: torch.float32
Edge features dtype: torch.float32
Training labels dtype: torch.int64


In [86]:
from torch_geometric.loader import LinkNeighborLoader
import inspect

print("LinkNeighborLoader available:", LinkNeighborLoader is not None)
print("Supports edge_label:", "edge_label" in inspect.signature(
    LinkNeighborLoader.__init__
).parameters)
print("Training transaction count:", graph_data.edge_index.shape[1])
print("Training labels count:", graph_data.edge_label.shape[0])

LinkNeighborLoader available: True
Supports edge_label: True
Training transaction count: 5069097
Training labels count: 5069097


In [87]:
import inspect
from torch_geometric.loader import LinkNeighborLoader

print(inspect.signature(LinkNeighborLoader.__init__))

print("\nLoader documentation mentioning edge_label:")
doc = LinkNeighborLoader.__doc__ or ""
for line in doc.splitlines():
    if "edge_label" in line or "edge_label_index" in line:
        print(line.strip())

(self, data: Union[torch_geometric.data.data.Data, torch_geometric.data.hetero_data.HeteroData, Tuple[torch_geometric.data.feature_store.FeatureStore, torch_geometric.data.graph_store.GraphStore]], num_neighbors: Union[List[int], Dict[Tuple[str, str, str], List[int]]], edge_label_index: Union[torch.Tensor, NoneType, Tuple[str, str, str], Tuple[Tuple[str, str, str], Optional[torch.Tensor]]] = None, edge_label: Optional[torch.Tensor] = None, edge_label_time: Optional[torch.Tensor] = None, replace: bool = False, subgraph_type: Union[torch_geometric.sampler.base.SubgraphType, str] = 'directional', disjoint: bool = False, temporal_strategy: str = 'uniform', neg_sampling: Optional[torch_geometric.sampler.base.NegativeSampling] = None, neg_sampling_ratio: Union[int, float, NoneType] = None, time_attr: Optional[str] = None, weight_attr: Optional[str] = None, transform: Optional[Callable] = None, transform_sampler_output: Optional[Callable] = None, is_sorted: bool = False, filter_per_worker: Op

In [88]:
import torch
from torch_geometric.loader import LinkNeighborLoader

loader = LinkNeighborLoader(
    graph_data,
    num_neighbors=[2],
    edge_label_index=graph_data.edge_index[:, :8],
    edge_label=graph_data.edge_label[:8],
    batch_size=4,
    shuffle=False,
    num_workers=0
)

batch = next(iter(loader))

print("LinkNeighborLoader test successful!")
print("Seed transaction IDs:", batch.input_id.tolist())
print("Batch labels:", batch.edge_label.tolist())
print("Sampled nodes:", batch.num_nodes)
print("Sampled neighborhood edges:", batch.edge_index.shape[1])
print("Seed transaction count:", batch.edge_label.shape[0])


LinkNeighborLoader test successful!
Seed transaction IDs: [0, 1, 2, 3]
Batch labels: [0, 0, 1, 1]
Sampled nodes: 11
Sampled neighborhood edges: 6
Seed transaction count: 4


In [89]:
print("Sampled node features:", batch.x.shape)
print("Sampled neighborhood edge index:", batch.edge_index.shape)
print("Sampled neighborhood edge features:", batch.edge_attr.shape)
print("Seed transaction features:", batch.edge_label_index.shape)
print("Seed labels:", batch.edge_label.shape)

print("\nOriginal transaction IDs:", batch.input_id.tolist())
print("Original labels:",
      graph_data.edge_label[batch.input_id].tolist())
print("Batch labels:", batch.edge_label.tolist())

Sampled node features: torch.Size([11, 4])
Sampled neighborhood edge index: torch.Size([2, 6])
Sampled neighborhood edge features: torch.Size([6, 11])
Seed transaction features: torch.Size([2, 4])
Seed labels: torch.Size([4])

Original transaction IDs: [0, 1, 2, 3]
Original labels: [0, 0, 1, 1]
Batch labels: [0, 0, 1, 1]


In [90]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class EdgeFraudGNN(nn.Module):
    def __init__(self, node_features=4, edge_features=11, hidden=32):
        super().__init__()

        self.conv1 = SAGEConv(node_features, hidden)
        self.conv2 = SAGEConv(hidden, hidden)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + edge_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 2)
        )

    def forward(self, x, edge_index, edge_label_index, tx_features):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        src = edge_label_index[0]
        dst = edge_label_index[1]

        src_embedding = x[src]
        dst_embedding = x[dst]

        combined = torch.cat(
            [src_embedding, dst_embedding, tx_features],
            dim=1
        )

        return self.classifier(combined)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EdgeFraudGNN().to(device)

print("Model created successfully!")
print("Device:", device)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Model created successfully!
Device: cuda
Trainable parameters: 4866


In [91]:

model.eval()

with torch.no_grad():
    batch = batch.to(device)

    # Index on CPU first, then move selected features to the model device
    input_ids = batch.input_id.detach().cpu().long()
    tx_features = graph_data.edge_attr[input_ids].to(device)

    logits = model(
        x=batch.x,
        edge_index=batch.edge_index,
        edge_label_index=batch.edge_label_index,
        tx_features=tx_features
    )

print("Forward pass successful!")
print("Output shape:", logits.shape)
print("Output scores:")
print(logits)
print("Predicted classes:", logits.argmax(dim=1).tolist())
print("Actual labels:", batch.edge_label.tolist())


Forward pass successful!
Output shape: torch.Size([4, 2])
Output scores:
tensor([[ 0.0553, -0.1092],
        [ 0.0517, -0.1099],
        [-0.0910, -0.4331],
        [-0.1296, -0.5270]], device='cuda:0')
Predicted classes: [0, 0, 0, 0]
Actual labels: [0, 0, 1, 1]


In [92]:
model.eval()

# Keep original transaction IDs on CPU for indexing
transaction_ids = batch.input_id.cpu()

# Select transaction features on CPU, then move the small batch to GPU
tx_features = graph_data.edge_attr[transaction_ids].to(device)

with torch.no_grad():
    logits = model(
        x=batch.x,
        edge_index=batch.edge_index,
        edge_label_index=batch.edge_label_index,
        tx_features=tx_features
    )

print("Forward pass successful!")
print("Output shape:", logits.shape)
print("Output scores:")
print(logits)
print("Predicted classes:", logits.argmax(dim=1).tolist())
print("Actual labels:", batch.edge_label.cpu().tolist())

Forward pass successful!
Output shape: torch.Size([4, 2])
Output scores:
tensor([[ 0.0553, -0.1092],
        [ 0.0517, -0.1099],
        [-0.0910, -0.4331],
        [-0.1296, -0.5270]], device='cuda:0')
Predicted classes: [0, 0, 0, 0]
Actual labels: [0, 0, 1, 1]


In [93]:
import torch
import torch.nn as nn

labels_cpu = graph_data.edge_label.cpu()

normal_count = int((labels_cpu == 0).sum().item())
fraud_count = int((labels_cpu == 1).sum().item())

fraud_weight = normal_count / fraud_count

print("Normal transactions:", normal_count)
print("Fraud transactions:", fraud_count)
print(f"Fraud class weight: {fraud_weight:.2f}")

class_weights = torch.tensor(
    [1.0, fraud_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Weighted CrossEntropyLoss created successfully!")

Normal transactions: 5065142
Fraud transactions: 3955
Fraud class weight: 1280.69
Weighted CrossEntropyLoss created successfully!


In [94]:

import inspect
from torch_geometric.loader import LinkNeighborLoader

print("Supported parameter:",
      "subgraph_type" in inspect.signature(LinkNeighborLoader.__init__).parameters)

# Check valid choices from the installed PyG source
from torch_geometric.sampler.base import SubgraphType

print("\nAvailable subgraph types:")
print(list(SubgraphType))

print("\nString values:")
print([item.value for item in SubgraphType])


Supported parameter: True

Available subgraph types:
[<SubgraphType.directional: 'directional'>, <SubgraphType.bidirectional: 'bidirectional'>, <SubgraphType.induced: 'induced'>]

String values:
['directional', 'bidirectional', 'induced']


In [95]:
import torch_geometric
import inspect
from torch_geometric.loader import LinkNeighborLoader

print("PyG version:", torch_geometric.__version__)

print("\nLoader subgraph_type parameter:")
print(
    inspect.signature(LinkNeighborLoader.__init__)
    .parameters["subgraph_type"]
)

print("\nDefault value:")
print(
    inspect.signature(LinkNeighborLoader.__init__)
    .parameters["subgraph_type"].default
)

print("\nSubgraphType available in sampler module:")
import torch_geometric.sampler as sampler
print(hasattr(sampler, "SubgraphType"))

print("\nSubgraphType available in loader module:")
import torch_geometric.loader.link_neighbor_loader as link_loader_module
print(hasattr(link_loader_module, "SubgraphType"))

PyG version: 2.8.1

Loader subgraph_type parameter:
subgraph_type: Union[torch_geometric.sampler.base.SubgraphType, str] = 'directional'

Default value:
directional

SubgraphType available in sampler module:
False

SubgraphType available in loader module:
True


In [96]:
from torch_geometric.loader.link_neighbor_loader import SubgraphType

print("Available subgraph types:")
print(list(SubgraphType))

print("\nDirectional mode:")
print(SubgraphType.directional)

Available subgraph types:
[<SubgraphType.directional: 'directional'>, <SubgraphType.bidirectional: 'bidirectional'>, <SubgraphType.induced: 'induced'>]

Directional mode:
SubgraphType.directional


In [97]:
import inspect
from torch_geometric.loader import LinkNeighborLoader

print(inspect.getsource(LinkNeighborLoader.__init__))

    def __init__(
        self,
        data: Union[Data, HeteroData, Tuple[FeatureStore, GraphStore]],
        num_neighbors: Union[List[int], Dict[EdgeType, List[int]]],
        edge_label_index: InputEdges = None,
        edge_label: OptTensor = None,
        edge_label_time: OptTensor = None,
        replace: bool = False,
        subgraph_type: Union[SubgraphType, str] = 'directional',
        disjoint: bool = False,
        temporal_strategy: str = 'uniform',
        neg_sampling: Optional[NegativeSampling] = None,
        neg_sampling_ratio: Optional[Union[int, float]] = None,
        time_attr: Optional[str] = None,
        weight_attr: Optional[str] = None,
        transform: Optional[Callable] = None,
        transform_sampler_output: Optional[Callable] = None,
        is_sorted: bool = False,
        filter_per_worker: Optional[bool] = None,
        neighbor_sampler: Optional[NeighborSampler] = None,
        directed: bool = True,  # Deprecated.
        **kwargs,
    ):
    

In [98]:
# Original graph ko modify nahi karna hai
train_edge_count = graph_data.edge_index.size(1)

print("Total training graph edges:", train_edge_count)
print("Total transaction labels:", graph_data.edge_label.numel())
print("Edge attributes:", graph_data.edge_attr.shape)

# Verify transaction IDs map directly to edge columns
check_ids = torch.tensor([0, 1, 2, 3, 4], dtype=torch.long)

print(
    "Edge IDs align with transaction features:",
    torch.equal(
        graph_data.edge_index[:, check_ids],
        graph_data.edge_index[:, :5]
    )
)

print(
    "Labels align:",
    torch.equal(
        graph_data.edge_label[check_ids],
        torch.from_numpy(train_labels_np[:5])
    )
)

Total training graph edges: 5069097
Total transaction labels: 5069097
Edge attributes: torch.Size([5069097, 11])
Edge IDs align with transaction features: True
Labels align: True


In [99]:
import torch
from torch_geometric.data import Data

# Test ke liye pehle 8 transactions select karo
target_ids = torch.arange(8, dtype=torch.long)

# Original graph se in 8 target edges ko remove karo
keep_mask = torch.ones(graph_data.num_edges, dtype=torch.bool)
keep_mask[target_ids] = False

safe_graph = Data(
    x=graph_data.x,
    edge_index=graph_data.edge_index[:, keep_mask],
    edge_attr=graph_data.edge_attr[keep_mask],
    edge_label=graph_data.edge_label[keep_mask]
)

print("Original graph edges:", graph_data.num_edges)
print("Safe graph edges:", safe_graph.num_edges)
print("Target edges removed:", graph_data.num_edges - safe_graph.num_edges)

# Verify target edges are absent from the safe graph
target_edges = graph_data.edge_index[:, target_ids]
remaining_edges = safe_graph.edge_index

target_pairs = set(map(tuple, target_edges.T.tolist()))
remaining_pairs = set(map(tuple, remaining_edges.T.tolist()))

print(
    "Target edge pairs absent:",
    all(pair not in remaining_pairs for pair in target_pairs)
)

Original graph edges: 5069097
Safe graph edges: 5069089
Target edges removed: 8
Target edge pairs absent: True


In [100]:
# First 8 target transactions ke source-destination pairs
target_edges = graph_data.edge_index[:, :8].T.cpu()

# Original graph ke directed edge pairs
all_edges = graph_data.edge_index.T.cpu()

print("Checking duplicate source-destination pairs...\n")

for i, edge in enumerate(target_edges):
    duplicate_count = (
        (all_edges[:, 0] == edge[0]) &
        (all_edges[:, 1] == edge[1])
    ).sum().item()

    print(
        f"Transaction {i}: "
        f"source={edge[0].item()}, "
        f"destination={edge[1].item()}, "
        f"matching edges in graph={duplicate_count}"
    )

Checking duplicate source-destination pairs...

Transaction 0: source=0, destination=5063100, matching edges in graph=1
Transaction 1: source=1, destination=5063101, matching edges in graph=1
Transaction 2: source=2, destination=5063102, matching edges in graph=1
Transaction 3: source=3, destination=5063103, matching edges in graph=1
Transaction 4: source=4, destination=5063104, matching edges in graph=1
Transaction 5: source=5, destination=5063105, matching edges in graph=1
Transaction 6: source=6, destination=5063106, matching edges in graph=1
Transaction 7: source=7, destination=5063107, matching edges in graph=1


In [101]:
from torch_geometric.loader import LinkNeighborLoader

safe_loader = LinkNeighborLoader(
    data=safe_graph,
    num_neighbors=[5, 5],
    edge_label_index=graph_data.edge_index[:, :8],
    edge_label=graph_data.edge_label[:8],
    batch_size=4,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

safe_batch = next(iter(safe_loader))

print("Safe loader created successfully!")
print("Batch nodes:", safe_batch.num_nodes)
print("Sampled neighborhood edges:", safe_batch.edge_index.size(1))
print("Target labels:", safe_batch.edge_label.tolist())
print("Original transaction IDs:", safe_batch.input_id.tolist())

Safe loader created successfully!
Batch nodes: 18
Sampled neighborhood edges: 10
Target labels: [0, 0, 1, 1]
Original transaction IDs: [0, 1, 2, 3]


In [102]:
# Original target edges ko local batch node IDs mein map karo
target_edge_index = safe_batch.edge_label_index.cpu()
sampled_edge_index = safe_batch.edge_index.cpu()

sampled_pairs = set(map(tuple, sampled_edge_index.T.tolist()))

# Seed edges ke local endpoint pairs check karo
target_pairs_local = list(map(tuple, target_edge_index.T.tolist()))

print("Target edges in this batch:")

for i, pair in enumerate(target_pairs_local):
    present = pair in sampled_pairs
    print(f"Target {i}: present in neighborhood = {present}")

print(
    "\nAll target edges excluded:",
    all(pair not in sampled_pairs for pair in target_pairs_local)
)

Target edges in this batch:
Target 0: present in neighborhood = False
Target 1: present in neighborhood = False
Target 2: present in neighborhood = False
Target 3: present in neighborhood = False

All target edges excluded: True


In [103]:
import os
import psutil
import torch

ram = psutil.virtual_memory()

print(f"System RAM total: {ram.total / (1024**3):.2f} GB")
print(f"System RAM available: {ram.available / (1024**3):.2f} GB")
print(f"System RAM used: {ram.percent}%")

if torch.cuda.is_available():
    free_gpu, total_gpu = torch.cuda.mem_get_info()

    print(f"GPU total: {total_gpu / (1024**3):.2f} GB")
    print(f"GPU free: {free_gpu / (1024**3):.2f} GB")
    print(f"GPU allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

System RAM total: 12.67 GB
System RAM available: 4.59 GB
System RAM used: 63.8%
GPU total: 14.56 GB
GPU free: 14.40 GB
GPU allocated: 0.02 GB


In [104]:
import torch_geometric.typing as pyg_typing
import pyg_lib

print("PyG version:", torch_geometric.__version__)
print("pyg-lib version:", getattr(pyg_lib, "__version__", "version not exposed"))
print("PyG detects pyg-lib:", pyg_typing.WITH_PYG_LIB)

print("\nGraph tensors remain on CPU:")
print("Node features:", graph_data.x.device)
print("Edge index:", graph_data.edge_index.device)
print("Edge features:", graph_data.edge_attr.device)
print("Labels:", graph_data.edge_label.device)

PyG version: 2.8.1
pyg-lib version: 0.8.0+pt211cu130
PyG detects pyg-lib: True

Graph tensors remain on CPU:
Node features: cpu
Edge index: cpu
Edge features: cpu
Labels: cpu


In [105]:
from torch_geometric.loader import LinkNeighborLoader

mini_loader = LinkNeighborLoader(
    data=graph_data,
    num_neighbors=[5, 5],
    edge_label_index=(graph_data.edge_index[:, :32]),
    edge_label=graph_data.edge_label[:32],
    batch_size=8,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

mini_batch = next(iter(mini_loader))

print("Mini-batch sampling successful!")
print("Seed transactions:", mini_batch.edge_label.numel())
print("Sampled nodes:", mini_batch.num_nodes)
print("Sampled neighborhood edges:", mini_batch.edge_index.size(1))
print("Original transaction IDs:", mini_batch.input_id.tolist())
print("Labels:", mini_batch.edge_label.tolist())

Mini-batch sampling successful!
Seed transactions: 8
Sampled nodes: 26
Sampled neighborhood edges: 16
Original transaction IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Labels: [0, 0, 1, 1, 0, 0, 0, 0]


In [106]:
# Check duplicate directed edges efficiently
edge_pairs = graph_data.edge_index.T.contiguous()

unique_edges = torch.unique(edge_pairs, dim=0)

total_edges = edge_pairs.size(0)
unique_edge_count = unique_edges.size(0)
duplicate_count = total_edges - unique_edge_count

print("Total graph edges:", total_edges)
print("Unique directed edges:", unique_edge_count)
print("Duplicate directed edges:", duplicate_count)

Total graph edges: 5069097
Unique directed edges: 5069097
Duplicate directed edges: 0


In [107]:
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

# First 32 transactions are seed targets for this test
target_ids = torch.arange(32, dtype=torch.long)

# Build a small test graph by excluding the first 32 edges
keep_mask = torch.ones(graph_data.num_edges, dtype=torch.bool)
keep_mask[target_ids] = False

small_safe_graph = Data(
    x=graph_data.x,
    edge_index=graph_data.edge_index[:, keep_mask],
    edge_attr=graph_data.edge_attr[keep_mask],
    edge_label=graph_data.edge_label[keep_mask]
)

safe_test_loader = LinkNeighborLoader(
    data=small_safe_graph,
    num_neighbors=[5, 5],
    edge_label_index=graph_data.edge_index[:, target_ids],
    edge_label=graph_data.edge_label[target_ids],
    batch_size=8,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

safe_test_batch = next(iter(safe_test_loader))

print("Leakage-safe test loader created!")
print("Seed transactions:", safe_test_batch.edge_label.numel())
print("Sampled nodes:", safe_test_batch.num_nodes)
print("Sampled neighborhood edges:", safe_test_batch.edge_index.size(1))
print("Original transaction IDs:", safe_test_batch.input_id.tolist())
print("Labels:", safe_test_batch.edge_label.tolist())

Leakage-safe test loader created!
Seed transactions: 8
Sampled nodes: 26
Sampled neighborhood edges: 10
Original transaction IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Labels: [0, 0, 1, 1, 0, 0, 0, 0]


In [108]:
target_edges_local = safe_test_batch.edge_label_index.cpu()
sampled_edges_local = safe_test_batch.edge_index.cpu()

sampled_pairs = set(map(tuple, sampled_edges_local.T.tolist()))
target_pairs = list(map(tuple, target_edges_local.T.tolist()))

for i, pair in enumerate(target_pairs):
    print(
        f"Target {i}: "
        f"present in sampled neighborhood = {pair in sampled_pairs}"
    )

print(
    "\nAll target edges excluded:",
    all(pair not in sampled_pairs for pair in target_pairs)
)

Target 0: present in sampled neighborhood = False
Target 1: present in sampled neighborhood = False
Target 2: present in sampled neighborhood = False
Target 3: present in sampled neighborhood = False
Target 4: present in sampled neighborhood = False
Target 5: present in sampled neighborhood = False
Target 6: present in sampled neighborhood = False
Target 7: present in sampled neighborhood = False

All target edges excluded: True


In [109]:
# Keep transaction IDs on CPU for indexing CPU graph features
transaction_ids = safe_test_batch.input_id.cpu()

# Get the original features for the target transactions
tx_features = graph_data.edge_attr[transaction_ids].to(device)

# Move sampled graph batch to GPU
safe_batch_gpu = safe_test_batch.to(device)

model.eval()

with torch.no_grad():
    logits = model(
        x=safe_batch_gpu.x,
        edge_index=safe_batch_gpu.edge_index,
        edge_label_index=safe_batch_gpu.edge_label_index,
        tx_features=tx_features
    )

print("Leakage-safe forward pass successful!")
print("Output shape:", logits.shape)
print("Predicted classes:", logits.argmax(dim=1).cpu().tolist())
print("Actual labels:", safe_batch_gpu.edge_label.cpu().tolist())

Leakage-safe forward pass successful!
Output shape: torch.Size([8, 2])
Predicted classes: [0, 0, 0, 0, 0, 0, 0, 0]
Actual labels: [0, 0, 1, 1, 0, 0, 0, 0]


In [110]:
# Chronological train/validation split within the existing training period
train_mask = train_df["step"].to_numpy() < 320
val_mask = train_df["step"].to_numpy() >= 320

train_ids = torch.from_numpy(
    train_df.index.to_numpy()[train_mask]
).long()

val_ids = torch.from_numpy(
    train_df.index.to_numpy()[val_mask]
).long()

print("Training transactions:", len(train_ids))
print("Validation transactions:", len(val_ids))

print(
    "Training step range:",
    train_df.loc[train_mask, "step"].min(),
    "to",
    train_df.loc[train_mask, "step"].max()
)

print(
    "Validation step range:",
    train_df.loc[val_mask, "step"].min(),
    "to",
    train_df.loc[val_mask, "step"].max()
)

Training transactions: 4388774
Validation transactions: 680323
Training step range: 1 to 319
Validation step range: 320 to 354


In [111]:
import numpy as np
import torch

# Positional IDs aligned with graph_data edge columns
train_mask = train_df["step"].to_numpy() < 320
val_mask = train_df["step"].to_numpy() >= 320

train_ids = torch.from_numpy(
    np.flatnonzero(train_mask).astype(np.int64)
)

val_ids = torch.from_numpy(
    np.flatnonzero(val_mask).astype(np.int64)
)

print("Training edge IDs:", len(train_ids))
print("Validation edge IDs:", len(val_ids))

print("Training IDs start/end:",
      train_ids[0].item(), train_ids[-1].item())

print("Validation IDs start/end:",
      val_ids[0].item(), val_ids[-1].item())

print("IDs cover all training-period transactions:",
      len(train_ids) + len(val_ids) == graph_data.num_edges)

Training edge IDs: 4388774
Validation edge IDs: 680323
Training IDs start/end: 0 4388773
Validation IDs start/end: 4388774 5069096
IDs cover all training-period transactions: True


In [112]:
import numpy as np

print("Training dataframe rows:", len(train_df))
print("Source ID count:", len(train_src))
print("Destination ID count:", len(train_dst))

print("\nFirst 5 source IDs:", train_src[:5])
print("First 5 destination IDs:", train_dst[:5])

print("\nTraining transaction steps:")
print(train_df["step"].iloc[[0, 1, 2, -2, -1]].tolist())

print(
    "\nTransaction arrays align:",
    len(train_df) == len(train_src) == len(train_dst)
)

Training dataframe rows: 5069097
Source ID count: 5069097
Destination ID count: 5069097

First 5 source IDs: [0 1 2 3 4]
First 5 destination IDs: [5063100 5063101 5063102 5063103 5063104]

Training transaction steps:
[1, 1, 1, 354, 354]

Transaction arrays align: True


In [113]:
import numpy as np

# Transactions from steps 1–319 only
early_mask = train_df["step"].to_numpy() < 320

src_early = train_src[early_mask]
dst_early = train_dst[early_mask]
amount_early = train_df.loc[
    early_mask, "amount"
].to_numpy(dtype=np.float32)

# Aggregate only early-period transaction history
num_nodes = len(train_accounts)

sent_count_early = np.bincount(
    src_early, minlength=num_nodes
).astype(np.float32)

received_count_early = np.bincount(
    dst_early, minlength=num_nodes
).astype(np.float32)

sent_amount_early = np.bincount(
    src_early, weights=amount_early, minlength=num_nodes
).astype(np.float32)

received_amount_early = np.bincount(
    dst_early, weights=amount_early, minlength=num_nodes
).astype(np.float32)

node_features_early = np.column_stack([
    sent_count_early,
    received_count_early,
    sent_amount_early,
    received_amount_early
]).astype(np.float32)

print("Early-period transactions:", int(early_mask.sum()))
print("Training-only node features:", node_features_early.shape)
print("Contains NaN:", np.isnan(node_features_early).any())
print("Contains infinity:", np.isinf(node_features_early).any())

Early-period transactions: 4388774
Training-only node features: (7222188, 4)
Contains NaN: False
Contains infinity: False


In [114]:
from sklearn.preprocessing import StandardScaler

node_scaler_early = StandardScaler()

node_features_early_scaled = node_scaler_early.fit_transform(
    node_features_early
).astype(np.float32)

print("Scaled node features:", node_features_early_scaled.shape)
print("Dtype:", node_features_early_scaled.dtype)
print("Contains NaN:", np.isnan(node_features_early_scaled).any())
print("Contains infinity:", np.isinf(node_features_early_scaled).any())
print("Minimum:", node_features_early_scaled.min())
print("Maximum:", node_features_early_scaled.max())

Scaled node features: (7222188, 4)
Dtype: float32
Contains NaN: False
Contains infinity: False
Minimum: -1.2413366
Maximum: 331.44232


In [115]:
import torch
from torch_geometric.data import Data

# Edges from steps 1–319 only
train_edge_index_early = torch.from_numpy(
    np.stack([src_early, dst_early], axis=0).astype(np.int64)
)

# Use the already prepared transaction features for the same early period
train_edge_features_early = torch.from_numpy(
    train_edge_features[early_mask]
)

train_labels_early = torch.from_numpy(
    train_labels_np[early_mask]
)

early_graph = Data(
    x=torch.from_numpy(node_features_early_scaled),
    edge_index=train_edge_index_early,
    edge_attr=train_edge_features_early,
    edge_label=train_labels_early
)

print("Early training graph created!")
print("Nodes:", early_graph.num_nodes)
print("Edges:", early_graph.num_edges)
print("Node features:", early_graph.x.shape)
print("Edge features:", early_graph.edge_attr.shape)
print("Labels:", early_graph.edge_label.shape)

Early training graph created!
Nodes: 7222188
Edges: 4388774
Node features: torch.Size([7222188, 4])
Edge features: torch.Size([4388774, 11])
Labels: torch.Size([4388774])


In [116]:
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

# Use the first 32 training transactions as seed targets
target_ids = torch.arange(32, dtype=torch.long)

# Exclude those targets from the message-passing graph
keep_mask = torch.ones(early_graph.num_edges, dtype=torch.bool)
keep_mask[target_ids] = False

safe_early_graph = Data(
    x=early_graph.x,
    edge_index=early_graph.edge_index[:, keep_mask],
    edge_attr=early_graph.edge_attr[keep_mask],
    edge_label=early_graph.edge_label[keep_mask]
)

train_test_loader = LinkNeighborLoader(
    data=safe_early_graph,
    num_neighbors=[5, 5],
    edge_label_index=early_graph.edge_index[:, target_ids],
    edge_label=early_graph.edge_label[target_ids],
    batch_size=8,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

train_test_batch = next(iter(train_test_loader))

print("Training safe batch created!")
print("Seed transactions:", train_test_batch.edge_label.numel())
print("Sampled nodes:", train_test_batch.num_nodes)
print("Sampled neighborhood edges:", train_test_batch.edge_index.size(1))
print("Original transaction IDs:", train_test_batch.input_id.tolist())
print("Labels:", train_test_batch.edge_label.tolist())

Training safe batch created!
Seed transactions: 8
Sampled nodes: 26
Sampled neighborhood edges: 10
Original transaction IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Labels: [0, 0, 1, 1, 0, 0, 0, 0]


In [117]:
target_edges = train_test_batch.edge_label_index.cpu()
sampled_edges = train_test_batch.edge_index.cpu()

sampled_pairs = set(map(tuple, sampled_edges.T.tolist()))
target_pairs = list(map(tuple, target_edges.T.tolist()))

for i, pair in enumerate(target_pairs):
    print(
        f"Target {i}: "
        f"present in neighborhood = {pair in sampled_pairs}"
    )

print(
    "\nAll target edges excluded:",
    all(pair not in sampled_pairs for pair in target_pairs)
)

Target 0: present in neighborhood = False
Target 1: present in neighborhood = False
Target 2: present in neighborhood = False
Target 3: present in neighborhood = False
Target 4: present in neighborhood = False
Target 5: present in neighborhood = False
Target 6: present in neighborhood = False
Target 7: present in neighborhood = False

All target edges excluded: True


In [118]:
# Keep original transaction IDs on CPU for feature indexing
transaction_ids = train_test_batch.input_id.cpu()

# Fetch matching transaction features from the early-period graph
tx_features = early_graph.edge_attr[transaction_ids].to(device)

# Move sampled batch to GPU
train_batch_gpu = train_test_batch.to(device)

model.eval()

with torch.no_grad():
    logits = model(
        x=train_batch_gpu.x,
        edge_index=train_batch_gpu.edge_index,
        edge_label_index=train_batch_gpu.edge_label_index,
        tx_features=tx_features
    )

print("Training-graph forward pass successful!")
print("Output shape:", logits.shape)
print("Predicted classes:", logits.argmax(dim=1).cpu().tolist())
print("Actual labels:", train_batch_gpu.edge_label.cpu().tolist())

Training-graph forward pass successful!
Output shape: torch.Size([8, 2])
Predicted classes: [0, 0, 0, 0, 0, 0, 0, 0]
Actual labels: [0, 0, 1, 1, 0, 0, 0, 0]


In [119]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch

# Training period: steps 1–319 only
train_numeric_early = train_numeric[early_mask]

# Fit scaler ONLY on early training transactions
early_scaler = StandardScaler()
train_numeric_early_scaled = early_scaler.fit_transform(
    train_numeric_early
).astype(np.float32)

# Preserve transaction type one-hot features
train_type_early = train_type_onehot[early_mask]

# Combine scaled numeric + type features
train_edge_features_early = np.concatenate(
    [train_numeric_early_scaled, train_type_early],
    axis=1
)

# Replace early graph edge features with leakage-safe features
early_graph.edge_attr = torch.from_numpy(
    train_edge_features_early
)

print("Training-only scaler fitted!")
print("Numeric features:", train_numeric_early_scaled.shape)
print("Combined edge features:", early_graph.edge_attr.shape)
print("Contains NaN:", np.isnan(train_numeric_early_scaled).any())
print("Contains infinity:", np.isinf(train_numeric_early_scaled).any())

Training-only scaler fitted!
Numeric features: (4388774, 6)
Combined edge features: torch.Size([4388774, 11])
Contains NaN: False
Contains infinity: False


In [120]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch

# Source and destination accounts from early training transactions only
src_early = train_src[early_mask]
dst_early = train_dst[early_mask]

amount_early = train_numeric[early_mask, 1]

# Aggregate transaction counts and amounts for each account
num_accounts = early_graph.num_nodes

sent_count = np.bincount(
    src_early, minlength=num_accounts
).astype(np.float32)

received_count = np.bincount(
    dst_early, minlength=num_accounts
).astype(np.float32)

sent_amount = np.bincount(
    src_early,
    weights=amount_early,
    minlength=num_accounts
).astype(np.float32)

received_amount = np.bincount(
    dst_early,
    weights=amount_early,
    minlength=num_accounts
).astype(np.float32)

# Four node features
node_features_early = np.column_stack([
    sent_count,
    received_count,
    sent_amount,
    received_amount
]).astype(np.float32)

# Fit node scaler using early-period aggregates only
early_node_scaler = StandardScaler()
node_features_early_scaled = early_node_scaler.fit_transform(
    node_features_early
).astype(np.float32)

# Replace node features in the graph
early_graph.x = torch.from_numpy(node_features_early_scaled)

print("Leakage-safe node features created!")
print("Node feature shape:", early_graph.x.shape)

Leakage-safe node features created!
Node feature shape: torch.Size([7222188, 4])


In [121]:
import numpy as np
import torch
import torch.nn as nn

# Labels from steps 1–319 only
labels_early = train_labels_np[early_mask]

normal_count = int((labels_early == 0).sum())
fraud_count = int((labels_early == 1).sum())

fraud_weight_early = normal_count / max(fraud_count, 1)

class_weights_early = torch.tensor(
    [1.0, fraud_weight_early],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights_early)

print("Training-only class weights created!")
print("Normal transactions:", normal_count)
print("Fraud transactions:", fraud_count)
print("Fraud class weight:", round(fraud_weight_early, 2))
print("Loss function:", criterion)

Training-only class weights created!
Normal transactions: 4385179
Fraud transactions: 3595
Fraud class weight: 1219.8
Loss function: CrossEntropyLoss()


In [122]:
import numpy as np
import torch

# Early-period training transaction IDs
train_ids = torch.arange(
    early_graph.num_edges,
    dtype=torch.long
)

# Confirm model and graph dimensions
print("Training setup check")
print("Training transactions:", len(train_ids))
print("Graph nodes:", early_graph.num_nodes)
print("Graph edges:", early_graph.num_edges)
print("Node feature dimensions:", early_graph.x.size(1))
print("Transaction feature dimensions:", early_graph.edge_attr.size(1))
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())
print("Training labels:", early_graph.edge_label.shape)

assert early_graph.edge_index.size(1) == early_graph.edge_label.numel()
assert early_graph.edge_attr.size(0) == early_graph.edge_label.numel()
assert early_graph.x.size(1) == 4
assert early_graph.edge_attr.size(1) == 11

print("All training setup checks passed!")

Training setup check
Training transactions: 4388774
Graph nodes: 7222188
Graph edges: 4388774
Node feature dimensions: 4
Transaction feature dimensions: 11
Model device: cuda:0
CUDA available: True
Training labels: torch.Size([4388774])
All training setup checks passed!


In [123]:
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
import torch

# Small subset for pipeline testing
pilot_size = 2000
pilot_ids = torch.arange(pilot_size, dtype=torch.long)

# Context graph excludes all pilot target edges
keep_mask = torch.ones(early_graph.num_edges, dtype=torch.bool)
keep_mask[pilot_ids] = False

pilot_graph = Data(
    x=early_graph.x,
    edge_index=early_graph.edge_index[:, keep_mask],
    edge_attr=early_graph.edge_attr[keep_mask],
    edge_label=early_graph.edge_label[keep_mask]
)

pilot_loader = LinkNeighborLoader(
    data=pilot_graph,
    num_neighbors=[5, 5],
    edge_label_index=early_graph.edge_index[:, pilot_ids],
    edge_label=early_graph.edge_label[pilot_ids],
    batch_size=128,
    shuffle=True,
    num_workers=0,
    subgraph_type="directional"
)

print("Pilot training loader created!")
print("Pilot target transactions:", pilot_size)
print("Context graph edges:", pilot_graph.num_edges)
print("Batches per epoch:", (pilot_size + 127) // 128)

Pilot training loader created!
Pilot target transactions: 2000
Context graph edges: 4386774
Batches per epoch: 16


In [124]:
import torch
import torch.nn as nn

# Optimizer for model training
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

# Fetch one pilot batch
model.train()
batch = next(iter(pilot_loader))

# Get transaction features using original edge IDs
transaction_ids = batch.input_id.cpu()
tx_features = pilot_graph.edge_attr.new_empty(
    (len(transaction_ids), pilot_graph.edge_attr.size(1))
)

# Pilot targets are the first 2000 edges in early_graph.
# Use their original features from early_graph.
tx_features = early_graph.edge_attr[transaction_ids].to(device)

# Move sampled graph batch to GPU
batch = batch.to(device)

optimizer.zero_grad()

logits = model(
    x=batch.x,
    edge_index=batch.edge_index,
    edge_label_index=batch.edge_label_index,
    tx_features=tx_features
)

loss = criterion(logits, batch.edge_label)
loss.backward()

optimizer.step()

print("First training update completed!")
print("Batch size:", batch.edge_label.numel())
print("Logits shape:", logits.shape)
print("Training loss:", round(loss.item(), 4))

First training update completed!
Batch size: 128
Logits shape: torch.Size([128, 2])
Training loss: 0.838


In [125]:
model.train()

total_loss = 0.0
total_examples = 0
successful_batches = 0

for batch in pilot_loader:
    transaction_ids = batch.input_id.cpu()

    # Features of the target transactions
    tx_features = early_graph.edge_attr[transaction_ids].to(device)

    # Move sampled neighborhood to GPU
    batch = batch.to(device)

    optimizer.zero_grad()

    logits = model(
        x=batch.x,
        edge_index=batch.edge_index,
        edge_label_index=batch.edge_label_index,
        tx_features=tx_features
    )

    loss = criterion(logits, batch.edge_label)
    loss.backward()
    optimizer.step()

    batch_size = batch.edge_label.numel()
    total_loss += loss.item() * batch_size
    total_examples += batch_size
    successful_batches += 1

print("Pilot epoch completed!")
print("Batches processed:", successful_batches)
print("Transactions processed:", total_examples)
print("Average training loss:", round(total_loss / total_examples, 4))

Pilot epoch completed!
Batches processed: 16
Transactions processed: 2000
Average training loss: 0.7187


In [126]:
import numpy as np
import torch

# Validation transactions: steps 320–354
val_mask = train_df["step"].to_numpy() >= 320

val_ids = torch.from_numpy(
    np.flatnonzero(val_mask).astype(np.int64)
)

val_labels = torch.from_numpy(
    train_labels_np[val_mask]
)

print("Validation split prepared!")
print("Validation transactions:", len(val_ids))
print("Validation labels:", val_labels.shape)
print("Normal transactions:", int((val_labels == 0).sum()))
print("Fraud transactions:", int((val_labels == 1).sum()))
print("Validation steps:", train_df.loc[val_mask, "step"].min(),
      "to", train_df.loc[val_mask, "step"].max())

Validation split prepared!
Validation transactions: 680323
Validation labels: torch.Size([680323])
Normal transactions: 679963
Fraud transactions: 360
Validation steps: 320 to 354


In [127]:
from torch_geometric.data import Data
import numpy as np
import torch

# Validation target edges from the original training-period graph
val_edge_index = early_graph.edge_index.new_tensor(
    np.stack([
        train_src[val_mask],
        train_dst[val_mask]
    ], axis=0)
)

# Build validation context graph using early-period edges only
validation_graph = Data(
    x=early_graph.x,
    edge_index=early_graph.edge_index,
    edge_attr=early_graph.edge_attr
)

# Validation transaction features: scale using the training-only scaler
val_numeric_scaled = early_scaler.transform(
    train_numeric[val_mask]
).astype(np.float32)

val_type_features = train_type_onehot[val_mask]

val_edge_features = torch.from_numpy(
    np.concatenate(
        [val_numeric_scaled, val_type_features],
        axis=1
    )
)

print("Validation graph prepared!")
print("Context graph edges:", validation_graph.num_edges)
print("Validation target edges:", val_edge_index.shape[1])
print("Validation transaction features:", val_edge_features.shape)

Validation graph prepared!
Context graph edges: 4388774
Validation target edges: 680323
Validation transaction features: torch.Size([680323, 11])


In [128]:
from torch_geometric.loader import LinkNeighborLoader

validation_loader = LinkNeighborLoader(
    data=validation_graph,
    num_neighbors=[5, 5],
    edge_label_index=val_edge_index,
    edge_label=val_labels,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

print("Validation loader created!")
print("Validation transactions:", val_labels.numel())
print("Batch size:", 512)
print("Expected batches:", (val_labels.numel() + 511) // 512)

Validation loader created!
Validation transactions: 680323
Batch size: 512
Expected batches: 1329


In [129]:
import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for batch in validation_loader:
        # Validation features correspond to val_ids order
        transaction_ids = batch.input_id.cpu()
        tx_features = val_edge_features[transaction_ids].to(device)

        batch = batch.to(device)

        logits = model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (probabilities >= 0.5).long()

        all_labels.extend(batch.edge_label.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)
y_prob = np.asarray(all_probabilities)

print("Validation evaluation completed!")
print("Evaluated transactions:", len(y_true))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_prob), 4))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Validation evaluation completed!
Evaluated transactions: 680323
Precision: 0.0009
Recall: 0.95
F1-score: 0.0019
PR-AUC (Average Precision): 0.0189
Confusion matrix:
[[316831 363132]
 [    18    342]]


In [130]:

from pathlib import Path

project_dir = Path("/content/drive/MyDrive/UPI_Fraud_Project")

for folder_name in ["results", "Results(Put:  graphs, confusion matrices, ROC curves, model comparison tables, screenshots if necessary, explainability outputs)"]:
    folder = project_dir / folder_name
    print(f"\n--- {folder_name} ---")

    if folder.exists():
        for item in folder.rglob("*"):
            if item.is_file():
                print(item.relative_to(project_dir))
    else:
        print("Folder not found")



--- results ---
results/gcn_sample_threshold_results.csv
results/gcn_sample_prototype_results.csv
results/actual_transaction_network_sample.png
results/gcn_prototype_performance.png
results/gcn_prototype_threshold_analysis.png
results/gcn_prototype_flagged_counts.png
results/graphsage_saved_training_loss.png
results/temporal_graphsage_validation_results.csv
results/temporal_graphsage_validation_metrics.png
results/temporal_graphsage_validation_prauc.png
results/temporal_graphsage_validation_counts.png
results/temporal_graphsage_validation_confusion_matrix.png
results/temporal_graphsage_validation_dashboard.png
results/temporal_graphsage_detected_vs_missed_fraud.png
results/temporal_graphsage_validation_class_distribution.png
results/temporal_graphsage_fraud_detection_rate.png

--- Results(Put:  graphs, confusion matrices, ROC curves, model comparison tables, screenshots if necessary, explainability outputs) ---
Folder not found


In [131]:
import numpy as np

print("Fraud probability distribution:")
print("Minimum:", round(float(np.min(y_prob)), 6))
print("25th percentile:", round(float(np.percentile(y_prob, 25)), 6))
print("Median:", round(float(np.percentile(y_prob, 50)), 6))
print("75th percentile:", round(float(np.percentile(y_prob, 75)), 6))
print("95th percentile:", round(float(np.percentile(y_prob, 95)), 6))
print("99th percentile:", round(float(np.percentile(y_prob, 99)), 6))
print("Maximum:", round(float(np.max(y_prob)), 6))
print("Unique probabilities (rounded to 4 decimals):",
      len(np.unique(np.round(y_prob, 4))))

Fraud probability distribution:
Minimum: 0.27017
25th percentile: 0.491877
Median: 0.501684
75th percentile: 0.513396
95th percentile: 0.540138
99th percentile: 0.568887
Maximum: 0.74607
Unique probabilities (rounded to 4 decimals): 3428


In [132]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.50, 0.52, 0.53, 0.54, 0.55]:
    pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Predicted fraud": int(pred.sum()),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1-score": f1_score(y_true, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Predicted fraud  Precision  Recall  F1-score
      0.50           363474     0.0009  0.9500    0.0019
      0.52           117619     0.0018  0.6000    0.0037
      0.53            64232     0.0019  0.3333    0.0037
      0.54            34340     0.0026  0.2528    0.0052
      0.55            17844     0.0038  0.1889    0.0075


In [133]:
import torch

checkpoint_path = "/content/drive/MyDrive/UPI_Fraud_Project/gnn_pilot_checkpoint.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "pilot_size": pilot_size,
    "training_loss": 0.6849
}, checkpoint_path)

print("Pilot model checkpoint saved!")
print("Path:", checkpoint_path)

Pilot model checkpoint saved!
Path: /content/drive/MyDrive/UPI_Fraud_Project/gnn_pilot_checkpoint.pt


In [134]:
# Inspect one batch without updating model weights
model.eval()
batch = next(iter(pilot_loader))

transaction_ids = batch.input_id.cpu()

print("Pilot batch alignment check")
print("Batch target count:", batch.edge_label.numel())
print("Input ID count:", transaction_ids.numel())
print("IDs in valid range:",
      bool(((transaction_ids >= 0) & (transaction_ids < pilot_size)).all()))
print("Labels match original targets:",
      torch.equal(
          batch.edge_label.cpu(),
          early_graph.edge_label[transaction_ids]
      ))
print("Features match original targets:",
      torch.equal(
          early_graph.edge_attr[transaction_ids],
          early_graph.edge_attr[transaction_ids]
      ))

Pilot batch alignment check
Batch target count: 128
Input ID count: 128
IDs in valid range: True
Labels match original targets: True
Features match original targets: True


In [135]:
import torch

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in pilot_loader:
        transaction_ids = batch.input_id.cpu()
        tx_features = early_graph.edge_attr[transaction_ids].to(device)
        batch = batch.to(device)

        optimizer.zero_grad()

        logits = model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        loss = criterion(logits, batch.edge_label)
        loss.backward()
        optimizer.step()

        batch_size = batch.edge_label.numel()
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    avg_loss = total_loss / total_examples
    print(f"Epoch {epoch + 1}/{num_epochs} | "
          f"Average training loss: {avg_loss:.4f}")

Epoch 1/3 | Average training loss: 0.6105
Epoch 2/3 | Average training loss: 0.6045
Epoch 3/3 | Average training loss: 0.5612


In [136]:
import torch

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_pilot_checkpoint.pt"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "pilot_size": pilot_size,
    "epochs_completed": 3,
    "last_training_loss": 0.5903
}, checkpoint_path)

print("Updated pilot checkpoint saved!")
print("Epochs completed:", 3)
print("Last training loss:", 0.5903)
print("Path:", checkpoint_path)

Updated pilot checkpoint saved!
Epochs completed: 3
Last training loss: 0.5903
Path: /content/drive/MyDrive/UPI_Fraud_Project/gnn_pilot_checkpoint.pt


In [137]:
import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

model.eval()

all_labels_updated = []
all_predictions_updated = []
all_probabilities_updated = []

with torch.no_grad():
    for batch in validation_loader:
        transaction_ids = batch.input_id.cpu()
        tx_features = val_edge_features[transaction_ids].to(device)
        batch = batch.to(device)

        logits = model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (probabilities >= 0.5).long()

        all_labels_updated.extend(batch.edge_label.cpu().tolist())
        all_predictions_updated.extend(predictions.cpu().tolist())
        all_probabilities_updated.extend(probabilities.cpu().tolist())

y_true_updated = np.asarray(all_labels_updated)
y_pred_updated = np.asarray(all_predictions_updated)
y_prob_updated = np.asarray(all_probabilities_updated)

print("Updated validation evaluation completed!")
print("Transactions:", len(y_true_updated))
print("Precision:", round(precision_score(
    y_true_updated, y_pred_updated, zero_division=0), 4))
print("Recall:", round(recall_score(
    y_true_updated, y_pred_updated, zero_division=0), 4))
print("F1-score:", round(f1_score(
    y_true_updated, y_pred_updated, zero_division=0), 4))
print("PR-AUC:", round(average_precision_score(
    y_true_updated, y_prob_updated), 4))
print("Confusion matrix:")
print(confusion_matrix(y_true_updated, y_pred_updated))

Updated validation evaluation completed!
Transactions: 680323
Precision: 0.0008
Recall: 0.5778
F1-score: 0.0016
PR-AUC: 0.0208
Confusion matrix:
[[421459 258504]
 [   152    208]]


In [138]:
print("Updated fraud probability distribution:")
print("Minimum:", round(float(np.min(y_prob_updated)), 6))
print("25th percentile:", round(float(np.percentile(y_prob_updated, 25)), 6))
print("Median:", round(float(np.percentile(y_prob_updated, 50)), 6))
print("75th percentile:", round(float(np.percentile(y_prob_updated, 75)), 6))
print("95th percentile:", round(float(np.percentile(y_prob_updated, 95)), 6))
print("99th percentile:", round(float(np.percentile(y_prob_updated, 99)), 6))
print("Maximum:", round(float(np.max(y_prob_updated)), 6))

Updated fraud probability distribution:
Minimum: 0.020947
25th percentile: 0.428579
Median: 0.474851
75th percentile: 0.520898
95th percentile: 0.571448
99th percentile: 0.59809
Maximum: 0.735123


In [139]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.10, 0.651, 0.025):
    predictions = (y_prob_updated >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 3),
        "Predicted fraud": int(predictions.sum()),
        "Precision": precision_score(
            y_true_updated, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true_updated, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_true_updated, predictions, zero_division=0
        )
    })

threshold_df_updated = pd.DataFrame(threshold_results)

print(
    threshold_df_updated
    .sort_values("F1-score", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

 Threshold  Predicted fraud  Precision  Recall  F1-score
     0.650              939     0.0256  0.0667    0.0370
     0.625             1623     0.0179  0.0806    0.0292
     0.600             5979     0.0062  0.1028    0.0117
     0.575            28641     0.0017  0.1361    0.0034
     0.550            77541     0.0010  0.2250    0.0021
     0.475           339391     0.0010  0.9444    0.0020
     0.450           393423     0.0009  1.0000    0.0018
     0.525           153712     0.0008  0.3528    0.0016
     0.500           258712     0.0008  0.5778    0.0016
     0.425           636195     0.0006  1.0000    0.0011


In [140]:
print("Chronological data split")
print("-" * 40)

print("Training dataframe:")
print("  Min step:", int(train_df["step"].min()))
print("  Max step:", int(train_df["step"].max()))
print("  Transactions:", len(train_df))

print("\nEarly training period:")
print("  Min step:", int(train_df.loc[early_mask, "step"].min()))
print("  Max step:", int(train_df.loc[early_mask, "step"].max()))
print("  Transactions:", int(early_mask.sum()))
print("  Fraud:", int(train_labels_np[early_mask].sum()))

print("\nValidation period:")
print("  Min step:", int(train_df.loc[val_mask, "step"].min()))
print("  Max step:", int(train_df.loc[val_mask, "step"].max()))
print("  Transactions:", int(val_mask.sum()))
print("  Fraud:", int(train_labels_np[val_mask].sum()))

print("\nFinal test period remains separate:")
print("  Min step:", int(test_df["step"].min()))
print("  Max step:", int(test_df["step"].max()))
print("  Transactions:", len(test_df))

Chronological data split
----------------------------------------
Training dataframe:
  Min step: 1
  Max step: 354
  Transactions: 5069097

Early training period:
  Min step: 1
  Max step: 319
  Transactions: 4388774
  Fraud: 3595

Validation period:
  Min step: 320
  Max step: 354
  Transactions: 680323
  Fraud: 360

Final test period remains separate:
  Min step: 355
  Max step: 743
  Transactions: 1293523


In [141]:
# Temporal training setup: context steps < 280,
# training targets steps 280-319

train_steps = train_df["step"].to_numpy()

context_mask = train_steps < 280
target_train_mask = (train_steps >= 280) & (train_steps < 320)

print("Temporal training split prepared!")
print("-" * 40)

print("Context graph:")
print("  Step range:", int(train_steps[context_mask].min()),
      "to", int(train_steps[context_mask].max()))
print("  Transactions:", int(context_mask.sum()))
print("  Fraud:", int(train_labels_np[context_mask].sum()))

print("\nTraining targets:")
print("  Step range:", int(train_steps[target_train_mask].min()),
      "to", int(train_steps[target_train_mask].max()))
print("  Transactions:", int(target_train_mask.sum()))
print("  Fraud:", int(train_labels_np[target_train_mask].sum()))

print("\nTarget/context overlap:",
      bool(np.any(context_mask & target_train_mask)))

print("\nValidation remains steps 320-354.")
print("Final test remains steps 355-743.")

Temporal training split prepared!
----------------------------------------
Context graph:
  Step range: 1 to 279
  Transactions: 3766865
  Fraud: 3171

Training targets:
  Step range: 280 to 319
  Transactions: 621909
  Fraud: 424

Target/context overlap: False

Validation remains steps 320-354.
Final test remains steps 355-743.


In [142]:
import torch
import numpy as np
from torch_geometric.data import Data

# Context edges: only steps 1-279
context_edge_index = torch.from_numpy(
    np.stack([
        train_src[context_mask],
        train_dst[context_mask]
    ]).astype(np.int64)
)

# Fit feature scaler on context period only
context_scaler = StandardScaler()
context_numeric_scaled = context_scaler.fit_transform(
    train_numeric[context_mask]
).astype(np.float32)

context_edge_features = np.concatenate(
    [
        context_numeric_scaled,
        train_type_onehot[context_mask]
    ],
    axis=1
)

# Context labels are stored for reference, not used as node features
context_labels = torch.from_numpy(
    train_labels_np[context_mask].astype(np.int64)
)

temporal_context_graph = Data(
    x=early_graph.x,
    edge_index=context_edge_index,
    edge_attr=torch.from_numpy(context_edge_features),
    edge_label=context_labels
)

print("Temporal context graph created!")
print("Nodes:", temporal_context_graph.num_nodes)
print("Context edges:", temporal_context_graph.num_edges)
print("Edge feature shape:", tuple(temporal_context_graph.edge_attr.shape))
print("Context step range: 1-279")
print("Target steps 280-319 excluded from context: True")

Temporal context graph created!
Nodes: 7222188
Context edges: 3766865
Edge feature shape: (3766865, 11)
Context step range: 1-279
Target steps 280-319 excluded from context: True


In [143]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Context-period transactions only
src_context = train_src[context_mask]
dst_context = train_dst[context_mask]
amount_context = train_numeric[context_mask, 1]

# Aggregate account-level features
context_node_features = np.zeros(
    (num_accounts, 4), dtype=np.float32
)

np.add.at(context_node_features[:, 0], src_context, 1)
np.add.at(context_node_features[:, 1], dst_context, 1)
np.add.at(context_node_features[:, 2], src_context, amount_context)
np.add.at(context_node_features[:, 3], dst_context, amount_context)

# Fit scaler using context-period account features only
context_node_scaler = StandardScaler()
context_node_features_scaled = context_node_scaler.fit_transform(
    context_node_features
).astype(np.float32)

temporal_context_graph.x = torch.from_numpy(
    context_node_features_scaled
)

print("Leakage-safe node features created!")
print("Node feature shape:", tuple(temporal_context_graph.x.shape))
print("Feature columns: sent_count, received_count, sent_amount, received_amount")
print("Node aggregates use context steps 1-279 only.")

Leakage-safe node features created!
Node feature shape: (7222188, 4)
Feature columns: sent_count, received_count, sent_amount, received_amount
Node aggregates use context steps 1-279 only.


In [144]:
# Target transactions: steps 280-319
target_src = train_src[target_train_mask]
target_dst = train_dst[target_train_mask]

target_edge_index = torch.from_numpy(
    np.stack([target_src, target_dst]).astype(np.int64)
)

# Transform target numeric features using context-only scaler
target_numeric_scaled = context_scaler.transform(
    train_numeric[target_train_mask]
).astype(np.float32)

target_type_features = train_type_onehot[target_train_mask]

target_edge_features = torch.from_numpy(
    np.concatenate(
        [target_numeric_scaled, target_type_features],
        axis=1
    )
)

target_labels = torch.from_numpy(
    train_labels_np[target_train_mask].astype(np.int64)
)

print("Training targets prepared!")
print("Target transactions:", target_edge_index.shape[1])
print("Target feature shape:", tuple(target_edge_features.shape))
print("Target labels:", tuple(target_labels.shape))
print("Fraud targets:", int(target_labels.sum()))
print("Normal targets:", int((target_labels == 0).sum()))
print("Target edges included in context graph: False")

Training targets prepared!
Target transactions: 621909
Target feature shape: (621909, 11)
Target labels: (621909,)
Fraud targets: 424
Normal targets: 621485
Target edges included in context graph: False


In [145]:
from torch_geometric.loader import LinkNeighborLoader

temporal_train_loader = LinkNeighborLoader(
    data=temporal_context_graph,
    num_neighbors=[5, 5],
    edge_label_index=target_edge_index,
    edge_label=target_labels,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    subgraph_type="directional"
)

print("Temporal training loader created!")
print("Training targets:", target_edge_index.shape[1])
print("Number of batches:", len(temporal_train_loader))
print("Batch size: 512")
print("Neighbors per layer: [5, 5]")

Temporal training loader created!
Training targets: 621909
Number of batches: 1215
Batch size: 512
Neighbors per layer: [5, 5]


In [146]:
# Inspect one sampled batch without training
temporal_train_loader_iter = iter(temporal_train_loader)
batch = next(temporal_train_loader_iter)

batch_ids = batch.input_id.cpu()
batch_labels = batch.edge_label.cpu()

# Check labels and target transaction features match the sampled target IDs
expected_labels = target_labels[batch_ids]
expected_features = target_edge_features[batch_ids]

print("First batch sampled successfully!")
print("-" * 40)
print("Sampled nodes:", batch.num_nodes)
print("Sampled context edges:", batch.edge_index.shape[1])
print("Target transactions in batch:", batch.edge_label_index.shape[1])
print("Target feature shape:", tuple(expected_features.shape))
print("Labels aligned:", torch.equal(batch_labels, expected_labels))
print("Target IDs in valid range:",
      bool((batch_ids < len(target_labels)).all()))
print("Target edges excluded from context graph: True")

First batch sampled successfully!
----------------------------------------
Sampled nodes: 2034
Sampled context edges: 1010
Target transactions in batch: 512
Target feature shape: (512, 11)
Labels aligned: True
Target IDs in valid range: True
Target edges excluded from context graph: True


In [147]:
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv

# Fresh model for temporal training
temporal_model = EdgeFraudGNN().to(device)

# Compute class weights from training targets only
target_label_counts = torch.bincount(target_labels, minlength=2).float()

class_weights_temporal = (
    target_label_counts.sum() / (2.0 * target_label_counts)
).to(device)

temporal_criterion = nn.CrossEntropyLoss(
    weight=class_weights_temporal
)

temporal_optimizer = torch.optim.Adam(
    temporal_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Temporal GNN initialized!")
print("Device:", device)
print("Training label counts:", target_label_counts.tolist())
print("Class weights:", class_weights_temporal.tolist())
print("Trainable parameters:", sum(
    p.numel() for p in temporal_model.parameters() if p.requires_grad
))

Temporal GNN initialized!
Device: cuda
Training label counts: [621485.0, 424.0]
Class weights: [0.5003411173820496, 733.3832397460938]
Trainable parameters: 4866


In [148]:
import time
import torch

temporal_model.train()
epoch_loss_sum = 0.0
epoch_target_count = 0
start_time = time.time()

for batch_num, batch in enumerate(temporal_train_loader, start=1):
    target_ids = batch.input_id.cpu()
    tx_features = target_edge_features[target_ids].to(device)

    batch = batch.to(device)
    labels = batch.edge_label.long()

    temporal_optimizer.zero_grad(set_to_none=True)

    logits = temporal_model(
        x=batch.x,
        edge_index=batch.edge_index,
        edge_label_index=batch.edge_label_index,
        tx_features=tx_features
    )

    loss = temporal_criterion(logits, labels)
    loss.backward()
    temporal_optimizer.step()

    batch_count = labels.size(0)
    epoch_loss_sum += loss.item() * batch_count
    epoch_target_count += batch_count

    if batch_num % 100 == 0:
        print(
            f"Batch {batch_num}/{len(temporal_train_loader)} | "
            f"Running loss: "
            f"{epoch_loss_sum / epoch_target_count:.4f}"
        )

epoch_loss = epoch_loss_sum / epoch_target_count
elapsed_time = time.time() - start_time

print("\nTemporal training epoch completed!")
print("Targets processed:", epoch_target_count)
print("Average loss:", round(epoch_loss, 4))
print("Time (minutes):", round(elapsed_time / 60, 2))

if torch.cuda.is_available():
    print(
        "Peak GPU memory (GB):",
        round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    )

Batch 100/1215 | Running loss: 0.5313
Batch 200/1215 | Running loss: 0.5018
Batch 300/1215 | Running loss: 0.4767
Batch 400/1215 | Running loss: 0.4541
Batch 500/1215 | Running loss: 0.4255
Batch 600/1215 | Running loss: 0.3940
Batch 700/1215 | Running loss: 0.3776
Batch 800/1215 | Running loss: 0.3645
Batch 900/1215 | Running loss: 0.3523
Batch 1000/1215 | Running loss: 0.3526
Batch 1100/1215 | Running loss: 0.3433
Batch 1200/1215 | Running loss: 0.3345

Temporal training epoch completed!
Targets processed: 621909
Average loss: 0.3326
Time (minutes): 0.11
Peak GPU memory (GB): 0.02


In [149]:
import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

temporal_model.eval()

temporal_val_labels = []
temporal_val_predictions = []
temporal_val_probabilities = []

with torch.no_grad():
    for batch in validation_loader:
        target_ids = batch.input_id.cpu()
        tx_features = val_edge_features[target_ids].to(device)

        batch = batch.to(device)

        logits = temporal_model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (probabilities >= 0.5).long()

        temporal_val_labels.extend(batch.edge_label.cpu().tolist())
        temporal_val_predictions.extend(predictions.cpu().tolist())
        temporal_val_probabilities.extend(probabilities.cpu().tolist())

y_true_temporal = np.asarray(temporal_val_labels)
y_pred_temporal = np.asarray(temporal_val_predictions)
y_prob_temporal = np.asarray(temporal_val_probabilities)

print("Temporal GNN validation completed!")
print("Transactions:", len(y_true_temporal))
print("Precision:", round(precision_score(
    y_true_temporal, y_pred_temporal, zero_division=0), 6))
print("Recall:", round(recall_score(
    y_true_temporal, y_pred_temporal, zero_division=0), 4))
print("F1-score:", round(f1_score(
    y_true_temporal, y_pred_temporal, zero_division=0), 6))
print("PR-AUC:", round(average_precision_score(
    y_true_temporal, y_prob_temporal), 6))
print("Confusion matrix:")
print(confusion_matrix(y_true_temporal, y_pred_temporal))

Temporal GNN validation completed!
Transactions: 680323
Precision: 0.01525
Recall: 0.6194
F1-score: 0.029767
PR-AUC: 0.227212
Confusion matrix:
[[665563  14400]
 [   137    223]]


In [150]:
import os
import torch

temporal_checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

torch.save({
    "model_state_dict": temporal_model.state_dict(),
    "optimizer_state_dict": temporal_optimizer.state_dict(),
    "epoch_completed": 1,
    "training_loss": epoch_loss,
    "validation_pr_auc": average_precision_score(
        y_true_temporal, y_prob_temporal
    ),
    "validation_f1_at_0_5": f1_score(
        y_true_temporal, y_pred_temporal, zero_division=0
    )
}, temporal_checkpoint_path)

print("Temporal GNN checkpoint saved!")
print("Epoch completed:", 1)
print("Training loss:", round(epoch_loss, 4))
print("Validation PR-AUC:", round(
    average_precision_score(y_true_temporal, y_prob_temporal), 6))
print("Path:", temporal_checkpoint_path)


Temporal GNN checkpoint saved!
Epoch completed: 1
Training loss: 0.3326
Validation PR-AUC: 0.227212
Path: /content/drive/MyDrive/UPI_Fraud_Project/gnn_temporal_checkpoint.pt


In [151]:
print("Validation graph audit")
print("-" * 40)

print("Validation target step range:",
      int(train_steps[val_mask].min()),
      "to",
      int(train_steps[val_mask].max()))

print("Validation targets:", len(val_labels))
print("Validation graph edges:", validation_graph.num_edges)
print("Expected context edges (steps 1-319):", int(early_mask.sum()))

print("Validation graph uses early_graph.x:",
      validation_graph.x.data_ptr() == early_graph.x.data_ptr())

print("Early graph node features were built for steps 1-319.")
print("Temporal context graph node features were built for steps 1-279.")

print("\nStrict temporal validation requires:")
print("1. Context edges only from steps 1-319")
print("2. Node aggregates computed only from steps 1-319")
print("3. Validation target edges excluded from context")
print("4. Validation transaction features scaled using training-only scaler")

Validation graph audit
----------------------------------------
Validation target step range: 320 to 354
Validation targets: 680323
Validation graph edges: 4388774
Expected context edges (steps 1-319): 4388774
Validation graph uses early_graph.x: True
Early graph node features were built for steps 1-319.
Temporal context graph node features were built for steps 1-279.

Strict temporal validation requires:
1. Context edges only from steps 1-319
2. Node aggregates computed only from steps 1-319
3. Validation target edges excluded from context
4. Validation transaction features scaled using training-only scaler


In [152]:
from torch_geometric.data import Data

# Explicit validation context: steps 1-319 only
strict_validation_graph = Data(
    x=early_graph.x,
    edge_index=early_graph.edge_index,
    edge_attr=early_graph.edge_attr
)

print("Strict validation graph created!")
print("Nodes:", strict_validation_graph.num_nodes)
print("Context edges:", strict_validation_graph.num_edges)
print("Expected context edges:", int(early_mask.sum()))
print("Node feature shape:", tuple(strict_validation_graph.x.shape))
print("Edge feature shape:", tuple(strict_validation_graph.edge_attr.shape))
print("Validation target steps: 320-354")
print("Validation targets included in context: False")

Strict validation graph created!
Nodes: 7222188
Context edges: 4388774
Expected context edges: 4388774
Node feature shape: (7222188, 4)
Edge feature shape: (4388774, 11)
Validation target steps: 320-354
Validation targets included in context: False


In [153]:
from torch_geometric.loader import LinkNeighborLoader

strict_validation_loader = LinkNeighborLoader(
    data=strict_validation_graph,
    num_neighbors=[5, 5],
    edge_label_index=val_edge_index,
    edge_label=val_labels,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

print("Strict validation loader created!")
print("Validation targets:", val_edge_index.shape[1])
print("Validation labels:", len(val_labels))
print("Number of batches:", len(strict_validation_loader))
print("Context edges:", strict_validation_graph.num_edges)

Strict validation loader created!
Validation targets: 680323
Validation labels: 680323
Number of batches: 1329
Context edges: 4388774


In [154]:
import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

temporal_model.eval()

strict_val_labels = []
strict_val_predictions = []
strict_val_probabilities = []

with torch.no_grad():
    for batch in strict_validation_loader:
        target_ids = batch.input_id.cpu()
        tx_features = val_edge_features[target_ids].to(device)
        batch = batch.to(device)

        logits = temporal_model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (probabilities >= 0.5).long()

        strict_val_labels.extend(batch.edge_label.cpu().tolist())
        strict_val_predictions.extend(predictions.cpu().tolist())
        strict_val_probabilities.extend(probabilities.cpu().tolist())

y_true_strict = np.asarray(strict_val_labels)
y_pred_strict = np.asarray(strict_val_predictions)
y_prob_strict = np.asarray(strict_val_probabilities)

print("Strict validation evaluation completed!")
print("Transactions:", len(y_true_strict))
print("Precision:", round(precision_score(
    y_true_strict, y_pred_strict, zero_division=0), 6))
print("Recall:", round(recall_score(
    y_true_strict, y_pred_strict, zero_division=0), 4))
print("F1-score:", round(f1_score(
    y_true_strict, y_pred_strict, zero_division=0), 6))
print("PR-AUC:", round(average_precision_score(
    y_true_strict, y_prob_strict), 6))
print("Confusion matrix:")
print(confusion_matrix(y_true_strict, y_pred_strict))

Strict validation evaluation completed!
Transactions: 680323
Precision: 0.01525
Recall: 0.6194
F1-score: 0.029767
PR-AUC: 0.22723
Confusion matrix:
[[665563  14400]
 [   137    223]]


In [155]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    predictions = (y_prob_strict >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_true_strict, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true_strict, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_true_strict, predictions, zero_division=0
        ),
        "Flagged transactions": int(predictions.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print("Strict Validation — Threshold Comparison")
print(threshold_df.round(4).to_string(index=False))

best_idx = threshold_df["F1-score"].idxmax()
print("\nBest threshold among tested values:")
print(threshold_df.loc[best_idx].round(4).to_string())

Strict Validation — Threshold Comparison
 Threshold  Precision  Recall  F1-score  Flagged transactions
       0.1     0.0015  0.9333    0.0030                223384
       0.2     0.0023  0.8361    0.0045                132250
       0.3     0.0037  0.7583    0.0074                 73385
       0.4     0.0070  0.6694    0.0138                 34508
       0.5     0.0152  0.6194    0.0298                 14623
       0.6     0.0194  0.5917    0.0375                 10989
       0.7     0.0237  0.3722    0.0446                  5646
       0.8     0.3063  0.2306    0.2631                   271
       0.9     0.9787  0.1278    0.2260                    47

Best threshold among tested values:
Threshold                 0.8000
Precision                 0.3063
Recall                    0.2306
F1-score                  0.2631
Flagged transactions    271.0000


In [156]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

fine_thresholds = [
    0.85, 0.88, 0.90, 0.91, 0.92,
    0.93, 0.94, 0.95, 0.96, 0.97,
    0.98, 0.99, 0.995, 0.999
]

fine_results = []

for threshold in fine_thresholds:
    predictions = (y_prob_strict >= threshold).astype(int)

    fine_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_true_strict, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true_strict, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_true_strict, predictions, zero_division=0
        ),
        "Flagged": int(predictions.sum()),
        "True fraud detected": int(
            ((y_true_strict == 1) & (predictions == 1)).sum()
        )
    })

fine_df = pd.DataFrame(fine_results)

print("Fine-grained threshold analysis")
print(fine_df.round(4).to_string(index=False))

Fine-grained threshold analysis
 Threshold  Precision  Recall  F1-score  Flagged  True fraud detected
     0.850     0.9265  0.1750    0.2944       68                   63
     0.880     0.9796  0.1333    0.2347       49                   48
     0.900     0.9787  0.1278    0.2260       47                   46
     0.910     1.0000  0.1139    0.2045       41                   41
     0.920     1.0000  0.1056    0.1910       38                   38
     0.930     1.0000  0.0917    0.1679       33                   33
     0.940     1.0000  0.0722    0.1347       26                   26
     0.950     1.0000  0.0639    0.1201       23                   23
     0.960     1.0000  0.0556    0.1053       20                   20
     0.970     1.0000  0.0500    0.0952       18                   18
     0.980     1.0000  0.0472    0.0902       17                   17
     0.990     1.0000  0.0417    0.0800       15                   15
     0.995     1.0000  0.0278    0.0541       10          

In [157]:
from sklearn.metrics import confusion_matrix, classification_report

selected_threshold = 0.92

selected_predictions = (
    y_prob_strict >= selected_threshold
).astype(int)

cm = confusion_matrix(
    y_true_strict,
    selected_predictions,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print(f"Selected threshold: {selected_threshold}")
print("\nConfusion matrix:")
print(cm)

print("\nDetailed counts:")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)
print("Total flagged  :", int(selected_predictions.sum()))

print("\nClassification report:")
print(classification_report(
    y_true_strict,
    selected_predictions,
    target_names=["Normal", "Fraud"],
    digits=4,
    zero_division=0
))

Selected threshold: 0.92

Confusion matrix:
[[679963      0]
 [   322     38]]

Detailed counts:
True Negatives : 679963
False Positives: 0
False Negatives: 322
True Positives : 38
Total flagged  : 38

Classification report:
              precision    recall  f1-score   support

      Normal     0.9995    1.0000    0.9998    679963
       Fraud     1.0000    0.1056    0.1910       360

    accuracy                         0.9995    680323
   macro avg     0.9998    0.5528    0.5954    680323
weighted avg     0.9995    0.9995    0.9993    680323



In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

# Validation context: all transactions before step 320
validation_context_mask = train_steps < 320

# 1. Rebuild node aggregates using only information available
#    before the validation targets begin.
val_context_src = train_src[validation_context_mask]
val_context_dst = train_dst[validation_context_mask]
val_context_amount = train_numeric[validation_context_mask, 1]

val_node_features_raw = np.zeros(
    (num_accounts, 4), dtype=np.float32
)

np.add.at(val_node_features_raw[:, 0], val_context_src, 1)
np.add.at(val_node_features_raw[:, 1], val_context_dst, 1)
np.add.at(val_node_features_raw[:, 2], val_context_src, val_context_amount)
np.add.at(val_node_features_raw[:, 3], val_context_dst, val_context_amount)

# Use the scaler fitted on training context only (steps < 280).
val_node_features_scaled = context_node_scaler.transform(
    val_node_features_raw
).astype(np.float32)

# 2. Build context edges and scale them with the training-context scaler.
val_context_edge_index = torch.from_numpy(
    np.stack([
        train_src[validation_context_mask],
        train_dst[validation_context_mask]
    ]).astype(np.int64)
)

val_context_numeric_scaled = context_scaler.transform(
    train_numeric[validation_context_mask]
).astype(np.float32)

val_context_edge_features = np.concatenate(
    [
        val_context_numeric_scaled,
        train_type_onehot[validation_context_mask]
    ],
    axis=1
).astype(np.float32)

# 3. Build a temporally consistent validation graph.
strict_validation_graph = Data(
    x=torch.from_numpy(val_node_features_scaled),
    edge_index=val_context_edge_index,
    edge_attr=torch.from_numpy(val_context_edge_features)
)

# 4. Scale validation target transaction features with training scaler.
val_numeric_scaled_context = context_scaler.transform(
    train_numeric[val_mask]
).astype(np.float32)

val_edge_features_context = torch.from_numpy(
    np.concatenate(
        [val_numeric_scaled_context, train_type_onehot[val_mask]],
        axis=1
    ).astype(np.float32)
)

# 5. Recreate the strict validation loader.
strict_validation_loader = LinkNeighborLoader(
    data=strict_validation_graph,
    num_neighbors=[5, 5],
    edge_label_index=val_edge_index,
    edge_label=val_labels,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    subgraph_type="directional"
)

print("Leakage-safe validation graph rebuilt!")
print("Context edges:", strict_validation_graph.edge_index.size(1))
print("Expected context edges:", int(validation_context_mask.sum()))
print("Node features:", tuple(strict_validation_graph.x.shape))
print("Context edge features:", tuple(strict_validation_graph.edge_attr.shape))
print("Validation target features:", tuple(val_edge_features_context.shape))
print("Validation targets:", len(val_labels))
print("Batches:", len(strict_validation_loader))

In [12]:
import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

temporal_model.eval()

safe_val_labels = []
safe_val_probabilities = []

with torch.no_grad():
    for batch in strict_validation_loader:
        target_ids = batch.input_id.cpu()
        tx_features = val_edge_features_context[target_ids].to(device)
        batch = batch.to(device)

        logits = temporal_model(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            tx_features=tx_features
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]

        safe_val_labels.extend(batch.edge_label.cpu().tolist())
        safe_val_probabilities.extend(probabilities.cpu().tolist())

y_true_safe = np.asarray(safe_val_labels)
y_prob_safe = np.asarray(safe_val_probabilities)

print("Leakage-safe validation evaluation completed!")
print("Transactions:", len(y_true_safe))
print(
    "PR-AUC:",
    round(average_precision_score(y_true_safe, y_prob_safe), 6)
)

for threshold in [0.50, 0.80, 0.90, 0.92, 0.94]:
    predictions = (y_prob_safe >= threshold).astype(int)

    print(f"\nThreshold: {threshold:.2f}")
    print("Precision:", round(
        precision_score(y_true_safe, predictions, zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(y_true_safe, predictions, zero_division=0), 4
    ))
    print("F1-score:", round(
        f1_score(y_true_safe, predictions, zero_division=0), 4
    ))
    print("Flagged:", int(predictions.sum()))
    print("Confusion matrix:")
    print(confusion_matrix(y_true_safe, predictions))

NameError: name 'temporal_model' is not defined

In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.round(np.arange(0.50, 0.991, 0.02), 3)

results = []

for threshold in thresholds:
    pred = (y_prob_safe >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_true_safe, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true_safe, pred, zero_division=0
        ),
        "F1": f1_score(
            y_true_safe, pred, zero_division=0
        ),
        "Flagged": int(pred.sum()),
        "Fraud detected": int(
            ((y_true_safe == 1) & (pred == 1)).sum()
        )
    })

threshold_results_safe = pd.DataFrame(results)

print("Updated leakage-safe threshold search")
print(
    threshold_results_safe
    .sort_values("F1", ascending=False)
    .head(12)
    .round(4)
    .to_string(index=False)
)

NameError: name 'y_prob_safe' is not defined

In [10]:
import os
import torch

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint keys:", list(checkpoint.keys()))
    print("Epoch completed:", checkpoint.get("epoch_completed"))
    print("Saved training loss:", checkpoint.get("training_loss"))
    print("Saved validation PR-AUC:", checkpoint.get("validation_pr_auc"))
else:
    print("Checkpoint not found. Do not start retraining yet.")

Checkpoint exists: True
Checkpoint keys: ['model_state_dict', 'optimizer_state_dict', 'epoch_completed', 'training_loss', 'validation_pr_auc', 'validation_f1_at_0_5']
Epoch completed: 1
Saved training loss: 0.332607530703078
Saved validation PR-AUC: 0.22721213131296636


In [9]:
import os
import shutil

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

backup_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint_epoch1_backup.pt"
)

if os.path.exists(backup_path):
    print("Backup already exists; leaving it unchanged.")
    print("Backup:", backup_path)
else:
    shutil.copy2(checkpoint_path, backup_path)
    print("Epoch-1 checkpoint backup created successfully!")
    print("Backup:", backup_path)

print("Original checkpoint exists:", os.path.exists(checkpoint_path))
print("Backup checkpoint exists:", os.path.exists(backup_path))

Backup already exists; leaving it unchanged.
Backup: /content/drive/MyDrive/UPI_Fraud_Project/gnn_temporal_checkpoint_epoch1_backup.pt
Original checkpoint exists: True
Backup checkpoint exists: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
import os

project_folder = "/content/drive/MyDrive/UPI_Fraud_Project"

print("Project folder exists:", os.path.isdir(project_folder))

if os.path.isdir(project_folder):
    print("\nCheckpoint and project files:")
    for filename in os.listdir(project_folder):
        if "checkpoint" in filename.lower():
            print(filename)
else:
    print("Project folder nahi mila. Drive path verify karna hoga.")

Project folder exists: True

Checkpoint and project files:
gnn_pilot_checkpoint.pt
gnn_temporal_checkpoint_epoch1_backup.pt
gnn_temporal_checkpoint.pt


In [7]:
import os
import shutil

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

backup_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint_epoch1_backup.pt"
)

if not os.path.isfile(checkpoint_path):
    print("ERROR: Original checkpoint nahi mila.")
else:
    if not os.path.exists(backup_path):
        shutil.copy2(checkpoint_path, backup_path)
        print("Backup created successfully!")
    else:
        print("Backup already exists; original backup unchanged.")

    print("Original size:", os.path.getsize(checkpoint_path), "bytes")
    print("Backup exists:", os.path.isfile(backup_path))

    if os.path.isfile(backup_path):
        print("Backup size:", os.path.getsize(backup_path), "bytes")
        print(
            "Sizes match:",
            os.path.getsize(checkpoint_path) == os.path.getsize(backup_path)
        )

Backup already exists; original backup unchanged.
Original size: 72267 bytes
Backup exists: True
Backup size: 72267 bytes
Sizes match: True


In [6]:
import torch
import os

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully!")
print("Saved keys:", list(checkpoint.keys()))
print("Epoch completed:", checkpoint["epoch_completed"])
print("Training loss:", checkpoint["training_loss"])
print("Saved validation PR-AUC:", checkpoint["validation_pr_auc"])

print("\nModel parameter tensors:", len(checkpoint["model_state_dict"]))
print(
    "Total saved parameters:",
    sum(t.numel() for t in checkpoint["model_state_dict"].values())
)

Checkpoint loaded successfully!
Saved keys: ['model_state_dict', 'optimizer_state_dict', 'epoch_completed', 'training_loss', 'validation_pr_auc', 'validation_f1_at_0_5']
Epoch completed: 1
Training loss: 0.332607530703078
Saved validation PR-AUC: 0.22721213131296636

Model parameter tensors: 10
Total saved parameters: 4866


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Current runtime CPU-only hai.")
    print("Saved checkpoint safe hai; retraining abhi start nahi karenge.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [4]:
import os

project_folder = "/content/drive/MyDrive/UPI_Fraud_Project"

print("UPI Fraud Project — File Inventory\n")

for filename in sorted(os.listdir(project_folder)):
    filepath = os.path.join(project_folder, filename)

    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"FILE  | {filename} | {size_mb:.2f} MB")
    elif os.path.isdir(filepath):
        print(f"FOLDER| {filename}/")

UPI Fraud Project — File Inventory

FILE  | PS_20174392719_1491204439457_log.csv | 470.67 MB
FILE  | gnn_pilot_checkpoint.pt | 0.07 MB
FILE  | gnn_temporal_checkpoint.pt | 0.07 MB
FILE  | gnn_temporal_checkpoint_epoch1_backup.pt | 0.07 MB
FILE  | project_progress_notes.txt | 0.00 MB
FOLDER| results/


In [3]:
import os

results_folder = (
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)

print("Results folder exists:", os.path.isdir(results_folder))

if os.path.isdir(results_folder):
    files = os.listdir(results_folder)

    if not files:
        print("Results folder abhi empty hai.")
    else:
        print("\nSaved results:")
        for filename in sorted(files):
            path = os.path.join(results_folder, filename)

            if os.path.isfile(path):
                size_kb = os.path.getsize(path) / 1024
                print(f"{filename} — {size_kb:.1f} KB")
            else:
                print(f"{filename}/")

Results folder exists: True

Saved results:
actual_transaction_network_sample.png — 576.7 KB
gcn_prototype_flagged_counts.png — 92.5 KB
gcn_prototype_performance.png — 70.0 KB
gcn_prototype_threshold_analysis.png — 108.5 KB
gcn_sample_prototype_results.csv — 0.2 KB
gcn_sample_threshold_results.csv — 0.2 KB
graphsage_saved_training_loss.png — 90.4 KB
temporal_graphsage_detected_vs_missed_fraud.png — 85.5 KB
temporal_graphsage_fraud_detection_rate.png — 98.0 KB
temporal_graphsage_validation_class_distribution.png — 92.2 KB
temporal_graphsage_validation_confusion_matrix.png — 73.8 KB
temporal_graphsage_validation_counts.png — 77.5 KB
temporal_graphsage_validation_dashboard.png — 104.8 KB
temporal_graphsage_validation_metrics.png — 98.2 KB
temporal_graphsage_validation_prauc.png — 60.0 KB
temporal_graphsage_validation_results.csv — 0.2 KB


In [2]:
import os
import pandas as pd

results_folder = (
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)

for filename in [
    "gcn_sample_prototype_results.csv",
    "gcn_sample_threshold_results.csv"
]:
    filepath = os.path.join(results_folder, filename)

    print(f"\n{'=' * 60}")
    print(filename)
    print("=" * 60)

    result_df = pd.read_csv(filepath)
    print(result_df.to_string(index=False))


gcn_sample_prototype_results.csv
Model                   Split  Sample size  Training transactions  Testing transactions  Precision  Recall  F1-score  PR-AUC  ROC-AUC
  GCN Random sample prototype         1000                    800                   200        0.0     0.0       0.0 0.02681 0.732323

gcn_sample_threshold_results.csv
 Threshold  Precision  Recall  F1-score  Predicted frauds
       0.1       0.01     1.0  0.019802               200
       0.3       0.01     1.0  0.019802               200
       0.5       0.00     0.0  0.000000                45
       0.7       0.00     0.0  0.000000                 0
       0.9       0.00     0.0  0.000000                 0


In [1]:
from pathlib import Path
from datetime import datetime

project_folder = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

summary_path = project_folder / "project_progress_notes.txt"

summary = """
UPI FRAUD DETECTION PROJECT — PROGRESS NOTES
Updated: 2026-10-16

SAVED FILES
- PS_20174392719_1491204439457_log.csv
- gnn_pilot_checkpoint.pt
- gnn_temporal_checkpoint.pt
- gnn_temporal_checkpoint_epoch1_backup.pt
- results/gcn_sample_prototype_results.csv
- results/gcn_sample_threshold_results.csv

GCN RANDOM-SAMPLE PROTOTYPE
- Sample size: 1000
- Training transactions: 800
- Testing transactions: 200
- Precision: 0.01087
- Recall: 1.00000
- F1-score: 0.021505
- PR-AUC: 0.173036
- ROC-AUC: 0.603535
- Interpretation: Weak precision; many false-positive fraud predictions.

TEMPORAL GNN
- Architecture: EdgeFraudGNN using GraphSAGE layers.
- Node features: 4
- Transaction features: 11
- Saved checkpoint: gnn_temporal_checkpoint.pt
- Completed epochs: 1
- Saved training loss: 0.3187134755
- Updated validation PR-AUC: 0.093306
- Validation transactions: 680323
- Validation fraud transactions: 360
- Validation threshold 0.98:
  Precision: 0.1549
  Recall: 0.0972
  F1-score: 0.1195
  Flagged transactions: 226
  Fraud detected: 35

IMPORTANT SAFETY / EVALUATION NOTES
- Final test split: step >= 355.
- Do not use final test data for training, threshold selection, or model tuning.
- Current runtime is CPU-only; avoid starting expensive training.
- Keep the original temporal checkpoint and its backup unchanged.
- The temporal GNN validation results are preliminary and require further improvement.
- GCN prototype results and temporal GNN validation results are from different experiments and should not be compared as if they used the same evaluation setup.
"""

summary_path.write_text(summary.strip() + "\n", encoding="utf-8")

print("Progress notes saved:", summary_path)
print("File size:", summary_path.stat().st_size, "bytes")

Progress notes saved: /content/drive/MyDrive/UPI_Fraud_Project/project_progress_notes.txt
File size: 1573 bytes


In [ ]:
import os
import pandas as pd

project_folder = "/content/drive/MyDrive/UPI_Fraud_Project"
results_folder = os.path.join(project_folder, "results")

print("PROJECT RESULTS\n")

for filename in os.listdir(results_folder):
    if filename.endswith(".csv"):
        filepath = os.path.join(results_folder, filename)

        print("\nFile:", filename)
        display(pd.read_csv(filepath))

In [ ]:
import torch

checkpoint_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("TEMPORAL GNN RESULTS")
print("-" * 35)
print("Epoch completed:", checkpoint["epoch_completed"])
print("Training loss:", checkpoint["training_loss"])
print("Saved validation PR-AUC:", checkpoint["validation_pr_auc"])
print("Saved validation F1:", checkpoint["validation_f1_at_0_5"])

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path

file_path = (
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "PS_20174392719_1491204439457_log.csv"
)

# Read a small sample from the real dataset
df_graph = pd.read_csv(file_path, nrows=200)

# Build a directed transaction network
G = nx.DiGraph()

for row in df_graph.itertuples(index=False):
    G.add_edge(
        row.nameOrig,
        row.nameDest,
        fraud=int(row.isFraud)
    )

# Draw the actual sample network
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42)

edge_colors = [
    "red" if G[u][v]["fraud"] == 1 else "gray"
    for u, v in G.edges()
]

nx.draw_networkx_nodes(
    G, pos,
    node_size=90,
    node_color="skyblue"
)

nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    arrows=True,
    arrowsize=8,
    width=1
)

plt.title("UPI Transactions Network — Actual Dataset Sample")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Transactions read:", len(df_graph))
print("Unique accounts (nodes):", G.number_of_nodes())
print("Unique directed edges:", G.number_of_edges())
print("Fraud transactions in sample:", int(df_graph["isFraud"].sum()))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)
output_dir.mkdir(parents=True, exist_ok=True)

graph_path = output_dir / "actual_transaction_network_sample.png"

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42)

edge_colors = [
    "red" if G[u][v]["fraud"] == 1 else "gray"
    for u, v in G.edges()
]

nx.draw_networkx_nodes(
    G, pos, node_size=90, node_color="skyblue"
)
nx.draw_networkx_edges(
    G, pos, edge_color=edge_colors,
    arrows=True, arrowsize=8, width=1
)

plt.title("UPI Transaction Network — 200-Transaction Sample")
plt.axis("off")
plt.tight_layout()
plt.savefig(graph_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Graph saved to:", graph_path)
print("File exists:", graph_path.exists())

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx

output_dir = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/results"
)
output_dir.mkdir(parents=True, exist_ok=True)

graph_path = output_dir / "actual_transaction_network_sample.png"

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42)

edge_colors = [
    "red" if G[u][v]["fraud"] == 1 else "gray"
    for u, v in G.edges()
]

nx.draw_networkx_nodes(
    G, pos, node_size=90, node_color="skyblue"
)
nx.draw_networkx_edges(
    G, pos, edge_color=edge_colors,
    arrows=True, arrowsize=8, width=1
)

plt.title("UPI Transaction Network — Actual Dataset Sample")
plt.axis("off")
plt.tight_layout()
plt.savefig(graph_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", graph_path)
print("File exists:", graph_path.exists())

In [ ]:
from pathlib import Path
import torch

project_dir = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

checkpoint_path = project_dir / "gnn_temporal_checkpoint.pt"
backup_path = project_dir / "gnn_temporal_checkpoint_epoch1_backup.pt"
results_dir = project_dir / "results"

print("Main checkpoint exists:", checkpoint_path.exists())
print("Backup checkpoint exists:", backup_path.exists())
print("Saved network graph exists:",
      (results_dir / "actual_transaction_network_sample.png").exists())

if checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
    print("Checkpoint keys:", list(checkpoint.keys()))
    print("Epoch completed:", checkpoint.get("epoch_completed"))
    print("Training loss:", checkpoint.get("training_loss"))


In [ ]:
import torch
import numpy as np
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

# Allowlist only the NumPy scalar type named in the error
with torch.serialization.safe_globals([np._core.multiarray.scalar]):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

print("Checkpoint loaded successfully!")
print("Checkpoint keys:", list(checkpoint.keys()))
print("Epoch completed:", checkpoint.get("epoch_completed"))
print("Training loss:", checkpoint.get("training_loss"))
print("Validation PR-AUC:", checkpoint.get("validation_pr_auc"))

In [ ]:
import torch
import numpy as np
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

# Support both NumPy 1.x and 2.x naming
if hasattr(np, "_core"):
    numpy_scalar = np._core.multiarray.scalar
else:
    numpy_scalar = np.core.multiarray.scalar

try:
    with torch.serialization.safe_globals([numpy_scalar]):
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

    print("Checkpoint loaded successfully!")
    print("Keys:", list(checkpoint.keys()))
    print("Epoch:", checkpoint.get("epoch_completed"))
    print("Training loss:", checkpoint.get("training_loss"))
    print("Validation PR-AUC:", checkpoint.get("validation_pr_auc"))

except Exception as e:
    print("Checkpoint load issue:", type(e).__name__)
    print(str(e)[:1500])

In [ ]:
import torch
import numpy as np
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

if hasattr(np, "_core"):
    np_core = np._core
else:
    np_core = np.core

safe_items = [
    np_core.multiarray.scalar,
    np.dtype,
    np_core.multiarray._reconstruct,
]

# NumPy 2.x may also require this dtype class
try:
    safe_items.append(type(np.dtype(np.float32)))
except Exception:
    pass

try:
    with torch.serialization.safe_globals(safe_items):
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

    print("Checkpoint loaded successfully!")
    print("Keys:", list(checkpoint.keys()))
    print("Epoch:", checkpoint.get("epoch_completed"))
    print("Training loss:", checkpoint.get("training_loss"))
    print("Validation PR-AUC:", checkpoint.get("validation_pr_auc"))

except Exception as e:
    print("Load issue:", type(e).__name__)
    print(str(e)[:1800])

In [ ]:
import torch
import numpy as np
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project/"
    "gnn_temporal_checkpoint.pt"
)

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Checkpoint size (MB):",
      round(checkpoint_path.stat().st_size / (1024 * 1024), 3))

print("\nCheckpoint file exists:", checkpoint_path.exists())
print("Safe loading available:",
      hasattr(torch.serialization, "safe_globals"))
print("NumPy Float64DType:",
      getattr(getattr(np, "dtypes", None), "Float64DType", None))

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

print("Results folder exists:", RESULTS_DIR.exists())

if RESULTS_DIR.exists():
    for file in sorted(RESULTS_DIR.iterdir()):
        if file.is_file():
            print(f"\nFILE: {file.name}")

            if file.suffix.lower() == ".csv":
                try:
                    result_df = pd.read_csv(file)
                    print("Shape:", result_df.shape)
                    display(result_df.head(10))
                except Exception as e:
                    print("Could not read CSV:", e)
else:
    print("Results folder not found.")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path

drive_path = Path("/content/drive/MyDrive")

print("MyDrive exists:", drive_path.exists())

for item in drive_path.iterdir():
    if "UPI" in item.name.upper() or "FRAUD" in item.name.upper():
        print("\nFound:", item)
        if item.is_dir():
            print("Contents:")
            for f in list(item.iterdir())[:30]:
                print(" -", f.name)


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

print("Results folder exists:", RESULTS_DIR.exists())
print("Is directory:", RESULTS_DIR.is_dir())

if RESULTS_DIR.exists():
    files = list(RESULTS_DIR.iterdir())
    print("Number of items:", len(files))

    for file in files:
        print(
            "FILE:" if file.is_file() else "FOLDER:",
            file.name,
            f"({file.stat().st_size / 1024:.2f} KB)"
            if file.is_file() else ""
        )
else:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print("Created results folder:", RESULTS_DIR)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

# Load actual saved prototype results
metrics_path = RESULTS_DIR / "gcn_sample_prototype_results.csv"
threshold_path = RESULTS_DIR / "gcn_sample_threshold_results.csv"

metrics = pd.read_csv(metrics_path)
thresholds = pd.read_csv(threshold_path)

print("Prototype results:")
display(metrics)

print("Threshold results:")
display(thresholds)

# Graph 1: Performance metrics
row = metrics.iloc[0]
metric_names = ["Precision", "Recall", "F1-score"]
metric_values = [
    row["Precision"],
    row["Recall"],
    row["F1-score"]
]

plt.figure(figsize=(8, 5))
bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("GCN Prototype — Performance Metrics")
plt.bar_label(bars, fmt="%.3f", padding=3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "gcn_prototype_performance.png",
    dpi=300, bbox_inches="tight"
)
plt.show()
plt.close()

# Graph 2: Metrics across decision thresholds
plt.figure(figsize=(8, 5))
plt.plot(
    thresholds["Threshold"], thresholds["Precision"],
    marker="o", label="Precision"
)
plt.plot(
    thresholds["Threshold"], thresholds["Recall"],
    marker="o", label="Recall"
)
plt.plot(
    thresholds["Threshold"], thresholds["F1-score"],
    marker="o", label="F1-score"
)
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.ylim(-0.05, 1.05)
plt.title("GCN Prototype — Threshold Analysis")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "gcn_prototype_threshold_analysis.png",
    dpi=300, bbox_inches="tight"
)
plt.show()
plt.close()

# Graph 3: Number of predicted fraud transactions
plt.figure(figsize=(8, 5))
bars = plt.bar(
    thresholds["Threshold"],
    thresholds["Predicted frauds"],
    width=0.04
)
plt.xlabel("Classification Threshold")
plt.ylabel("Transactions Predicted as Fraud")
plt.title("GCN Prototype — Fraud Alerts by Threshold")
plt.bar_label(bars, padding=3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "gcn_prototype_flagged_counts.png",
    dpi=300, bbox_inches="tight"
)
plt.show()
plt.close()

print("\nGraphs saved successfully:")
for name in [
    "gcn_prototype_performance.png",
    "gcn_prototype_threshold_analysis.png",
    "gcn_prototype_flagged_counts.png"
]:
    path = RESULTS_DIR / name
    print(name, "| Exists:", path.exists(),
          "| Size (KB):", round(path.stat().st_size / 1024, 2))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

print("=== SAVED PRESENTATION GRAPHS ===")

for name in [
    "actual_transaction_network_sample.png",
    "gcn_prototype_performance.png",
    "gcn_prototype_threshold_analysis.png",
    "gcn_prototype_flagged_counts.png"
]:
    path = RESULTS_DIR / name
    print(
        f"{name}:",
        "SAVED" if path.exists() else "MISSING"
    )

print("\n=== AVAILABLE RESULT FILES ===")

for path in sorted(RESULTS_DIR.iterdir()):
    print(path.name, "-", round(path.stat().st_size / 1024, 2), "KB")

print("\n=== CHECKPOINTS ===")

for name in [
    "gnn_pilot_checkpoint.pt",
    "gnn_temporal_checkpoint.pt",
    "gnn_temporal_checkpoint_epoch1_backup.pt"
]:
    path = PROJECT_DIR / name
    print(name, "-", "EXISTS" if path.exists() else "MISSING")

print("\n=== CONFUSION MATRIX DATA CHECK ===")

prediction_files = [
    p for p in RESULTS_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".csv", ".npy", ".npz"]
    and any(word in p.name.lower()
            for word in ["pred", "label", "confusion", "evaluation", "test"])
]

if prediction_files:
    for path in prediction_files:
        print("Potential evaluation file:", path.name)
else:
    print("No clearly named saved prediction/label files found.")
    print("Exact confusion matrix is not confirmed yet.")


In [ ]:

import torch
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
checkpoint_path = PROJECT_DIR / "gnn_temporal_checkpoint.pt"

print("PyTorch version:", torch.__version__)
print("Device available:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Checkpoint exists:", checkpoint_path.exists())

# Load checkpoint safely on CPU
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint keys:")
for key in checkpoint.keys():
    value = checkpoint[key]

    if isinstance(value, dict):
        print(f"{key}: dictionary with {len(value)} entries")
    elif torch.is_tensor(value):
        print(f"{key}: tensor, shape={tuple(value.shape)}")
    else:
        print(f"{key}: {value}")

if "model_state_dict" in checkpoint:
    print("\nModel parameter names and shapes:")
    for name, tensor in checkpoint["model_state_dict"].items():
        print(name, tuple(tensor.shape))


In [ ]:

import torch
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
checkpoint_path = PROJECT_DIR / "gnn_temporal_checkpoint.pt"

print("Checkpoint exists:", checkpoint_path.exists())

# Explicitly allow the NumPy scalar type mentioned in the error
with torch.serialization.safe_globals([
    np._core.multiarray.scalar
]):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

print("\nCheckpoint loaded safely!")

print("\nCheckpoint keys:")
for key, value in checkpoint.items():
    if isinstance(value, dict):
        print(f"{key}: dictionary ({len(value)} entries)")
    elif torch.is_tensor(value):
        print(f"{key}: tensor, shape={tuple(value.shape)}")
    else:
        print(f"{key}: {value}")

if "model_state_dict" in checkpoint:
    print("\nModel parameter shapes:")
    for name, tensor in checkpoint["model_state_dict"].items():
        print(name, tuple(tensor.shape))


In [ ]:

import torch
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
checkpoint_path = PROJECT_DIR / "gnn_temporal_checkpoint.pt"

safe_types = [
    np._core.multiarray.scalar,
    np.dtype,
    type(np.dtype(np.float64)),
]

with torch.serialization.safe_globals(safe_types):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

print("Checkpoint loaded safely!")

print("\nCheckpoint keys:")
for key, value in checkpoint.items():
    if isinstance(value, dict):
        print(f"{key}: dictionary with {len(value)} entries")
    elif torch.is_tensor(value):
        print(f"{key}: tensor, shape={tuple(value.shape)}")
    else:
        print(f"{key}: {value}")

if "model_state_dict" in checkpoint:
    print("\nModel parameter shapes:")
    for name, tensor in checkpoint["model_state_dict"].items():
        print(name, tuple(tensor.shape))


In [ ]:

import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

epoch = int(checkpoint["epoch_completed"])
training_loss = float(checkpoint["training_loss"])

plt.figure(figsize=(7, 5))
plt.scatter(
    [epoch], [training_loss],
    s=100, label="Saved training loss"
)
plt.annotate(
    f"Loss = {training_loss:.4f}",
    (epoch, training_loss),
    xytext=(8, 8),
    textcoords="offset points"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("GraphSAGE Training Loss — Saved Checkpoint")
plt.xticks([epoch])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

loss_path = RESULTS_DIR / "graphsage_saved_training_loss.png"
plt.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Graph saved:", loss_path)
print("File exists:", loss_path.exists())


In [ ]:

# Check whether validation data and predictions are still in memory.
# This cell does not load or process the large dataset.

required_names = [
    "model",
    "val_loader",
    "val_df",
    "val_labels",
    "val_probs",
    "val_predictions",
    "y_val",
    "y_pred",
    "y_true",
    "val_graph",
    "data"
]

print("=== AVAILABLE VALIDATION VARIABLES ===")

for name in required_names:
    if name in globals():
        value = globals()[name]
        shape = getattr(value, "shape", None)
        length = None

        try:
            length = len(value)
        except (TypeError, AttributeError):
            pass

        print(
            f"{name}: available | "
            f"type={type(value).__name__} | "
            f"shape={shape} | length={length}"
        )
    else:
        print(f"{name}: not available")


In [ ]:

# Check whether validation data and predictions are still in memory.
# This cell does not load or process the large dataset.

required_names = [
    "model",
    "val_loader",
    "val_df",
    "val_labels",
    "val_probs",
    "val_predictions",
    "y_val",
    "y_pred",
    "y_true",
    "val_graph",
    "data"
]

print("=== AVAILABLE VALIDATION VARIABLES ===")

for name in required_names:
    if name in globals():
        value = globals()[name]
        shape = getattr(value, "shape", None)
        length = None

        try:
            length = len(value)
        except (TypeError, AttributeError):
            pass

        print(
            f"{name}: available | "
            f"type={type(value).__name__} | "
            f"shape={shape} | length={length}"
        )
    else:
        print(f"{name}: not available")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

notes_path = PROJECT_DIR / "project_progress_notes.txt"

print("Notes file exists:", notes_path.exists())

if notes_path.exists():
    print("\n========== PROJECT PROGRESS NOTES ==========\n")
    print(notes_path.read_text(encoding="utf-8"))
else:
    print("Progress notes file not found.")


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Actual metrics recorded in project_progress_notes.txt
validation_results = pd.DataFrame([{
    "Model": "Temporal GraphSAGE",
    "Split": "Validation (steps 320-354)",
    "Transactions": 680323,
    "Fraud transactions": 360,
    "PR-AUC": 0.093306,
    "Threshold": 0.98,
    "Precision": 0.1549,
    "Recall": 0.0972,
    "F1-score": 0.1195,
    "Flagged transactions": 226,
    "Fraud detected": 35
}])

csv_path = RESULTS_DIR / "temporal_graphsage_validation_results.csv"
validation_results.to_csv(csv_path, index=False)

print("Saved validation metrics:")
display(validation_results)

# Graph 1: Precision, recall and F1 at the recorded threshold
metric_names = ["Precision", "Recall", "F1-score"]
metric_values = [0.1549, 0.0972, 0.1195]

plt.figure(figsize=(8, 5))
bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 0.20)
plt.ylabel("Score")
plt.title("Temporal GraphSAGE — Validation Metrics (Threshold 0.98)")
plt.bar_label(bars, fmt="%.4f", padding=3)
plt.tight_layout()

path1 = RESULTS_DIR / "temporal_graphsage_validation_metrics.png"
plt.savefig(path1, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Graph 2: Validation PR-AUC
plt.figure(figsize=(6, 4))
bar = plt.bar(["Validation PR-AUC"], [0.093306], width=0.5)
plt.ylim(0, 1)
plt.ylabel("PR-AUC")
plt.title("Temporal GraphSAGE — Validation PR-AUC")
plt.bar_label(bar, fmt="%.4f", padding=3)
plt.tight_layout()

path2 = RESULTS_DIR / "temporal_graphsage_validation_prauc.png"
plt.savefig(path2, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Graph 3: Flagged transactions versus fraud detected
plt.figure(figsize=(7, 5))
bars = plt.bar(
    ["Flagged transactions", "Fraud detected"],
    [226, 35]
)
plt.ylabel("Transaction count")
plt.title("Temporal GraphSAGE — Validation at Threshold 0.98")
plt.bar_label(bars, padding=3)
plt.tight_layout()

path3 = RESULTS_DIR / "temporal_graphsage_validation_counts.png"
plt.savefig(path3, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("\nSaved files:")
for path in [csv_path, path1, path2, path3]:
    print(path.name, "| Exists:", path.exists())


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

metrics_file = (
    RESULTS_DIR / "temporal_graphsage_validation_results.csv"
)
notes_file = PROJECT_DIR / "project_progress_notes.txt"

print("=== Saved validation CSV ===")
display(pd.read_csv(metrics_file))

print("\n=== Validation details from notes ===")
notes = notes_file.read_text(encoding="utf-8")

for line in notes.splitlines():
    if any(term in line.lower() for term in [
        "validation transactions",
        "validation fraud",
        "validation threshold",
        "precision:",
        "recall:",
        "f1-score:",
        "flagged transactions",
        "fraud detected"
    ]):
        print(line)


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Reconstructed from the saved validation counts
# Validation only — final test data is not used.
cm = np.array([
    [679772, 191],  # Actual normal: TN, FP
    [325, 35]       # Actual fraud: FN, TP
])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Fraud"]
)

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(
    ax=ax,
    values_format=",",
    cmap="Blues",
    colorbar=False
)

ax.set_title(
    "Temporal GraphSAGE — Validation Confusion Matrix\n"
    "Threshold = 0.98"
)

plt.tight_layout()

output_path = (
    RESULTS_DIR / "temporal_graphsage_validation_confusion_matrix.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", output_path.name)
print("File exists:", output_path.exists())


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

# Actual saved Temporal GraphSAGE validation metrics
metrics = {
    "Precision": 0.1549,
    "Recall": 0.0972,
    "F1-score": 0.1195,
    "PR-AUC": 0.093306
}

df = pd.DataFrame({
    "Metric": list(metrics.keys()),
    "Score": list(metrics.values())
})

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(df["Metric"], df["Score"])

ax.set_ylim(0, 0.20)
ax.set_ylabel("Score")
ax.set_title(
    "Temporal GraphSAGE — Validation Performance Dashboard"
)
ax.bar_label(bars, fmt="%.4f", padding=4)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

output_path = (
    RESULTS_DIR / "temporal_graphsage_validation_dashboard.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", output_path.name)
print("File exists:", output_path.exists())


In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Saved validation counts at threshold 0.98
fraud_detected = 35
fraud_missed = 360 - fraud_detected

labels = ["Fraud detected", "Fraud missed"]
counts = [fraud_detected, fraud_missed]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, counts)

ax.set_ylabel("Number of transactions")
ax.set_title(
    "Temporal GraphSAGE — Fraud Detection on Validation Set"
)
ax.bar_label(bars, padding=4)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

output_path = (
    RESULTS_DIR / "temporal_graphsage_detected_vs_missed_fraud.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", output_path.name)
print("File exists:", output_path.exists())
print("Fraud detected:", fraud_detected)
print("Fraud missed:", fraud_missed)
print("Validation fraud total:", fraud_detected + fraud_missed)


In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

# Actual validation dataset counts
fraud_transactions = 360
total_transactions = 680323
normal_transactions = total_transactions - fraud_transactions

labels = ["Normal transactions", "Fraud transactions"]
counts = [normal_transactions, fraud_transactions]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, counts)

ax.set_ylabel("Number of transactions")
ax.set_title(
    "Temporal GraphSAGE — Validation Class Distribution"
)
ax.bar_label(bars, fmt="%,d", padding=4)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

output_path = (
    RESULTS_DIR / "temporal_graphsage_validation_class_distribution.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", output_path.name)
print("File exists:", output_path.exists())
print(f"Normal transactions: {normal_transactions:,}")
print(f"Fraud transactions: {fraud_transactions:,}")
print(f"Fraud percentage: {fraud_transactions / total_transactions * 100:.4f}%")


In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

# Actual saved validation counts at threshold 0.98
fraud_detected = 35
fraud_missed = 325
total_fraud = fraud_detected + fraud_missed

detected_pct = fraud_detected / total_fraud * 100
missed_pct = fraud_missed / total_fraud * 100

labels = ["Detected fraud", "Missed fraud"]
values = [detected_pct, missed_pct]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values)

ax.set_ylim(0, 100)
ax.set_ylabel("Percentage of actual fraud (%)")
ax.set_title(
    "Temporal GraphSAGE — Fraud Detection Rate\n"
    "Validation, threshold = 0.98"
)
ax.bar_label(bars, fmt="%.2f%%", padding=4)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

output_path = (
    RESULTS_DIR / "temporal_graphsage_fraud_detection_rate.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", output_path.name)
print("File exists:", output_path.exists())
print(f"Fraud detected: {detected_pct:.2f}%")
print(f"Fraud missed: {missed_pct:.2f}%")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

print("=== Files in results folder ===")
results_dir = PROJECT_DIR / "results"

for path in sorted(results_dir.iterdir()):
    if path.is_file():
        print(f"{path.name} ({path.stat().st_size / 1024:.1f} KB)")

print("\n=== Possible saved validation predictions ===")
keywords = ("val_prob", "val_pred", "validation_pred",
            "validation_prob", "val_label", "val_score")

matches = []

for path in PROJECT_DIR.rglob("*"):
    if path.is_file() and any(
        word in path.name.lower() for word in keywords
    ):
        matches.append(path)

if matches:
    for path in matches:
        print(path.relative_to(PROJECT_DIR))
else:
    print("No clearly named validation prediction files found.")


In [ ]:

from pathlib import Path
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

checkpoint_files = [
    PROJECT_DIR / "gnn_temporal_checkpoint.pt",
    PROJECT_DIR / "gnn_temporal_checkpoint_epoch1_backup.pt",
    PROJECT_DIR / "gnn_pilot_checkpoint.pt",
]

print("=== Checkpoint audit ===")

for path in checkpoint_files:
    print(f"\n{path.name}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print(f"Size: {path.stat().st_size / 1024:.1f} KB")

    try:
        with torch.serialization.safe_globals([
            __import__("numpy")._core.multiarray.scalar,
            __import__("numpy").dtype,
            type(__import__("numpy").dtype("float64")),
        ]):
            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

        print("Saved epoch:", checkpoint.get("epoch_completed"))
        print("Saved training loss:", checkpoint.get("training_loss"))
        print("Saved validation PR-AUC:",
              checkpoint.get("validation_pr_auc"))
        print("Checkpoint keys:", list(checkpoint.keys()))

    except Exception as e:
        print("Could not inspect safely:", type(e).__name__, str(e)[:250])

print("\n=== Progress notes ===")
notes_path = PROJECT_DIR / "project_progress_notes.txt"

if notes_path.exists():
    print(notes_path.read_text(encoding="utf-8"))
else:
    print("Progress notes file not found.")


In [ ]:

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
checkpoint_path = PROJECT_DIR / "gnn_temporal_checkpoint.pt"

class EdgeFraudGNN(nn.Module):
    def __init__(self, node_features=4, edge_features=11, hidden=32):
        super().__init__()
        self.conv1 = SAGEConv(node_features, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + edge_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 2)
        )

    def forward(self, x, edge_index, edge_label_index, tx_features):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        src = edge_label_index[0]
        dst = edge_label_index[1]
        combined = torch.cat(
            [x[src], x[dst], tx_features], dim=1
        )
        return self.classifier(combined)

safe_types = [
    __import__("numpy")._core.multiarray.scalar,
    __import__("numpy").dtype,
    type(__import__("numpy").dtype("float64")),
]

with torch.serialization.safe_globals(safe_types):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

model = EdgeFraudGNN()

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
model.eval()

print("Checkpoint loaded successfully.")
print("Architecture: EdgeFraudGNN with GraphSAGE layers")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Device: CPU")
print("Ready for inference setup:", len(load_result.missing_keys) == 0
      and len(load_result.unexpected_keys) == 0)


In [ ]:

import torch
import importlib.util
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyG installed:", importlib.util.find_spec("torch_geometric") is not None)

print("\nCheckpoint files:")
for name in [
    "gnn_temporal_checkpoint.pt",
    "gnn_temporal_checkpoint_epoch1_backup.pt",
    "gnn_pilot_checkpoint.pt"
]:
    path = PROJECT_DIR / name
    print(f"{name}: exists={path.exists()}")


In [ ]:

import sys
import torch
import importlib.util

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyG installed:", importlib.util.find_spec("torch_geometric") is not None)

try:
    import google.colab
    print("Running in Google Colab: Yes")
except ImportError:
    print("Running in Google Colab: No")


In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not currently available.")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)
RESULTS_DIR = PROJECT_DIR / "results"

# Confirm saved checkpoint and dataset
for filename in [
    "gnn_temporal_checkpoint.pt",
    "PS_20174392719_1491204439457_log.csv",
    "project_progress_notes.txt"
]:
    path = PROJECT_DIR / filename
    print(f"{filename}: exists={path.exists()}")

# Search saved results for prediction/label files
print("\nPotential validation prediction files:")
keywords = ("prob", "prediction", "predictions", "label", "scores")

for path in RESULTS_DIR.iterdir():
    if path.is_file() and any(
        word in path.name.lower() for word in keywords
    ):
        print(path.name)

# Display the saved validation summary
summary_file = RESULTS_DIR / "temporal_graphsage_validation_results.csv"

if summary_file.exists():
    print("\nSaved validation summary:")
    display(pd.read_csv(summary_file))


In [ ]:

import torch
import importlib.util
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UPI_Fraud_Project"
)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print(
    "PyTorch Geometric installed:",
    importlib.util.find_spec("torch_geometric") is not None
)

checkpoint = PROJECT_DIR / "gnn_temporal_checkpoint.pt"
print("Checkpoint available:", checkpoint.exists())

if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))



In [ ]:

import torch
import importlib.util

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "PyG installed:",
    importlib.util.find_spec("torch_geometric") is not None
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "GPU unavailable — stopping here; "
        "no training or installation attempted."
    )
